# Dynamic Embedded Topic Model (DETM) for UN General Debates

## Overview
This notebook implements the Dynamic Embedded Topic Model from Dieng et al. (2019) "Topic Modeling in Embedding Spaces" applied to the UN General Debates corpus.

### Paper Reference
Dieng, A. B., Ruiz, F. J., & Blei, D. M. (2019). Topic modeling in embedding spaces. Transactions of the Association for Computational Linguistics, 7, 439-453.

### Model Architecture
**Generative Process:**
- Topic embeddings evolve over time: α_t ~ N(α_{t-1}, σ²I)
- Topic-word distributions: β_k,t = softmax(α_k,t · ρ) where ρ are word embeddings
- Document topic proportions: θ_t ~ N(0, I) (Gaussian prior)
- Word generation: w ~ Categorical(softmax(β^T θ))

**Variational Inference:**
- Approximate posterior: q(θ_t | w_t) = N(μ_t, Σ_t)
- Encoder network outputs μ_t and log σ_t from bag-of-words representation
- ELBO = E_q[log p(w|θ,α,ρ)] - KL(q(θ)||p(θ))

### Implementation Structure
1. **Data Processing**: Load, clean, and prepare UN debates corpus
   - UN-specific stopwords + WordNet lemmatization
   - IDF-ranked vocabulary selection (replaces raw-frequency ranking)
   - Tighter frequency gates (MIN_DF=30, MAX_DF=0.3)
2. **Embeddings**: Word2Vec skip-gram trained on corpus (matches Dieng et al.)
   - Replaces SentenceTransformer; W2V geometry is designed for inner-product factorisation
3. **Model Components**: Encoder, Decoder, Temporal Dynamics
4. **Loss Modules**: KL Divergence and Reconstruction Loss (modular)
5. **Training**: Optimization loop with ELBO maximization
6. **Evaluation**: Coherence, perplexity, topic quality metrics
7. **Visualization**: TensorBoard real-time monitoring (metrics, loss curves, gradients)


---
## 1. Setup and Imports

In [1]:
# Standard library imports
import os
import sys
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional, Union
from contextlib import nullcontext

# Load environment variables (Kaggle API credentials)
from dotenv import load_dotenv
load_dotenv()  # This loads variables from .env file

# Verify Kaggle credentials are loaded
if os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'):
    print("✓ Kaggle credentials loaded successfully")
else:
    print("⚠ Kaggle credentials not found. Make sure .env file contains KAGGLE_USERNAME and KAGGLE_KEY")

# Data processing
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from transformers import AutoTokenizer, AutoModel
import gensim
from gensim.models import Word2Vec
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Utilities
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE
import umap
import math
# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✓ All imports loaded successfully")


✓ Kaggle credentials loaded successfully
✓ All imports loaded successfully


class Config:
    """Central configuration for DETM experiment"""
    
    # Paths
    DATA_DIR = Path('../data')
    MODELS_DIR = Path('../models')
    OUTPUTS_DIR = Path('../outputs')
    
    # Data preprocessing — aligned with original DETM (Dieng et al. 2019)
    MIN_DF = 10       # Matches original; domain-specific words (ozone, deforestation) appear in fewer docs
    MAX_DF = 0.7      # Matches original; important topic markers (development, peace) appear frequently
    MAX_VOCAB_SIZE = 10000  # No longer used for selection (df gates control vocab); kept for compatibility
    MIN_DOC_LENGTH = 5   # Lowered for paragraph-level documents (paragraphs are shorter than full speeches)
    TRAIN_WORD_EMBEDDINGS = True   # Train W2V embeddings jointly — original default; helps topics separate
    
    # Model architecture
    NUM_TOPICS = 50  # K in the paper
    # Word2Vec skip-gram trained on corpus (replaces SentenceTransformer)
    # Skip-gram inner-product geometry directly matches DETM's β = softmax(α·ρ^T)
    EMBEDDING_DIM = 300  # Word2Vec standard dimension (was 384 for SentenceTransformer)
    W2V_WINDOW = 5       # Word2Vec context window
    W2V_EPOCHS = 10      # Word2Vec training epochs
    # NOTE: Word embeddings (ρ) are W2V skip-gram, raw (no L2-norm), TRAINABLE in DETMDecoder
    
    # Temporal Baseline Encoder (LSTM-based structured inference for η_t)
    COMPRESSION_DIM = 200  # Compression of BoW before LSTM (original: 200)
    LSTM_LAYERS = 3  # Number of LSTM layers (original: 3)
    LSTM_HIDDEN = 200  # Hidden units in each LSTM layer (original: 200)
    
    # Document Topic Encoder (Amortized inference for θ_d)
    DOC_HIDDEN_DIM = 800  # Hidden dimension for document encoder MLP
    DOC_DROPOUT = 0.0  # Dropout for document encoder (0.0 matches original — dropout in VAE encoders causes collapse)
    
    # Topic Embeddings (Mean-field inference for α^(t))
    # No neural network - just trainable parameters (T × K × L)
    INIT_ALPHA_STD = 1.0  # Initialization std for α parameters (matches original torch.randn default)
    # alpha_logvar initialized with randn in DETM.__init__ (matching original); this field unused
    INIT_ALPHA_LOGVAR = 0.0  # unused — see DETM.__init__
    
    # Legacy parameters (kept for compatibility)
    HIDDEN_DIM = 512  # Deprecated - use DOC_HIDDEN_DIM
    NUM_ENCODER_LAYERS = 2  # Deprecated
    DROPOUT = 0.2  # Deprecated - use DOC_DROPOUT
    
    # Temporal dynamics - Gaussian random walk prior variances (from Dieng et al. 2019)
    USE_TEMPORAL = True
    TEMPORAL_TYPE = 'gaussian'  # Deprecated
    ETA_PRIOR_VARIANCE   = 0.005  # δ²: prior variance for η random walk  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
    ALPHA_PRIOR_VARIANCE = 0.005  # γ²: prior variance for α random walk  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)
    TEMPORAL_VARIANCE = 0.005  # Deprecated alias - use ETA_PRIOR_VARIANCE or ALPHA_PRIOR_VARIANCE
    
    # Training — aligned with Dieng et al. 2019
    BATCH_SIZE = 700  # Original uses 1000; small batches destabilize global η LSTM
    NUM_EPOCHS = 400        # Paper trains longer; early stopping terminates when appropriate
    LEARNING_RATE = 0.0001  # Original paper default (--lr 0.0001)
    WEIGHT_DECAY = 1.2e-6   # From Dieng et al. 2019 (used with Adam, not AdamW)
    WARMUP_EPOCHS = 10      # Deprecated — no scheduler
    CLIP_GRAD = 0.0         # Disabled — matches original default (clip=0); set >0 to enable
    ANNEAL_PATIENCE = 10    # Anneal LR after this many consecutive non-improvement epochs
    ANNEAL_FACTOR = 4.0     # Divide LR by this amount on plateau (matches original ÷4)
    MIN_LR = 1e-6           # Stop training when LR drops below this threshold
    
    # Loss weights
    RECON_WEIGHT = 1.0  # Weight for reconstruction term
    KL_THETA_WEIGHT = 1.0  # Weight for KL(q(θ) || p(θ)) - document topics
    KL_ETA_WEIGHT = 1.0  # Weight for KL(q(η) || p(η)) - temporal baselines (normalized by D)
    KL_ALPHA_WEIGHT = 1.0  # Weight for KL(q(α) || p(α)) - topic embeddings (normalized by D)
    KL_WEIGHT = 1.0  # Deprecated - legacy total KL weight
    
    # Evaluation
    TOP_N_WORDS = [10, 15, 20]  # Top-N words for coherence
    COHERENCE_METRICS = ['c_v', 'c_npmi']  # Coherence types
    
    # Checkpointing
    SAVE_EVERY = 10  # Save checkpoint every N epochs
    PATIENCE = 15  # Early stopping patience
    
    def __init__(self):
        # Create directories if they don't exist
        self.DATA_DIR.mkdir(exist_ok=True)
        self.MODELS_DIR.mkdir(exist_ok=True)
        self.OUTPUTS_DIR.mkdir(exist_ok=True)

config = Config()
print("Configuration initialized successfully")
print(f"  MIN_DF={config.MIN_DF}, MAX_DF={config.MAX_DF}, EMBEDDING_DIM={config.EMBEDDING_DIM}")

In [ ]:
class Config:
    """Central configuration for DETM experiment"""
    
    # Paths
    DATA_DIR = Path('../data')
    MODELS_DIR = Path('../models')
    OUTPUTS_DIR = Path('../outputs')
    
    # Data preprocessing — aligned with original DETM (Dieng et al. 2019)
    MIN_DF = 10       # Matches original; domain-specific words (ozone, deforestation) appear in fewer docs
    MAX_DF = 0.7      # Matches original; important topic markers (development, peace) appear frequently
    MAX_VOCAB_SIZE = 10000  # No longer used for selection (df gates control vocab); kept for compatibility
    MIN_DOC_LENGTH = 5   # Lowered for paragraph-level documents (paragraphs are shorter than full speeches)
    TRAIN_WORD_EMBEDDINGS = True   # Train W2V embeddings jointly — original default; helps topics separate
    
    # Model architecture
    NUM_TOPICS = 50  # K in the paper
    # Word2Vec skip-gram trained on corpus (replaces SentenceTransformer)
    # Skip-gram inner-product geometry directly matches DETM's β = softmax(α·ρ^T)
    EMBEDDING_DIM = 300  # Word2Vec standard dimension (was 384 for SentenceTransformer)
    W2V_WINDOW = 5       # Word2Vec context window
    W2V_EPOCHS = 10      # Word2Vec training epochs
    # NOTE: Word embeddings (ρ) are W2V skip-gram, raw (no L2-norm), TRAINABLE in DETMDecoder
    
    # Temporal Baseline Encoder (LSTM-based structured inference for η_t)
    COMPRESSION_DIM = 200  # Compression of BoW before LSTM (original: 200)
    LSTM_LAYERS = 3  # Number of LSTM layers (original: 3)
    LSTM_HIDDEN = 200  # Hidden units in each LSTM layer (original: 200)
    
    # Document Topic Encoder (Amortized inference for θ_d)
    DOC_HIDDEN_DIM = 800  # Hidden dimension for document encoder MLP
    DOC_DROPOUT = 0.0  # Dropout for document encoder (0.0 matches original — dropout in VAE encoders causes collapse)
    
    # Topic Embeddings (Mean-field inference for α^(t))
    # No neural network - just trainable parameters (T × K × L)
    INIT_ALPHA_STD = 1.0  # Initialization std for α parameters (matches original torch.randn default)
    # alpha_logvar initialized with randn in DETM.__init__ (matching original); this field unused
    INIT_ALPHA_LOGVAR = 0.0  # unused — see DETM.__init__
    
    # Legacy parameters (kept for compatibility)
    HIDDEN_DIM = 512  # Deprecated - use DOC_HIDDEN_DIM
    NUM_ENCODER_LAYERS = 2  # Deprecated
    DROPOUT = 0.2  # Deprecated - use DOC_DROPOUT
    
    # Temporal dynamics - Gaussian random walk prior variances (from Dieng et al. 2019)
    USE_TEMPORAL = True
    TEMPORAL_TYPE = 'gaussian'  # Deprecated
    ETA_PRIOR_VARIANCE   = 0.005  # δ²: prior variance for η random walk  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
    ALPHA_PRIOR_VARIANCE = 0.005  # γ²: prior variance for α random walk  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)
    TEMPORAL_VARIANCE = 0.005  # Deprecated alias - use ETA_PRIOR_VARIANCE or ALPHA_PRIOR_VARIANCE
 
    # Training — aligned with Dieng et al. 2019
    BATCH_SIZE = 700  # Original uses 1000; small batches destabilize global η LSTM
    NUM_EPOCHS = 1000       # Headroom for slow convergence; early stopping + LR annealing terminate early
    LEARNING_RATE = 0.0001  # Matches original paper default (adjidieng/DETM --lr 0.0001)
    WEIGHT_DECAY = 1.2e-6   # From Dieng et al. 2019 (used with Adam, not AdamW)
    WARMUP_EPOCHS = 10      # Deprecated — no scheduler
    CLIP_GRAD = 0.0         # Disabled — matches original default (clip=0 per Dieng et al. 2019)
    ANNEAL_PATIENCE = 10    # Anneal LR after this many consecutive non-improvement epochs
    ANNEAL_FACTOR = 4.0     # Divide LR by this amount on plateau (matches original ÷4)
    MIN_LR = 1e-6           # Stop training when LR drops below this threshold
    
    # Evaluation
    TOP_N_WORDS = [10, 15, 20]  # Top-N words for coherence
    COHERENCE_METRICS = ['c_v', 'c_npmi']  # Coherence types
    
    # Checkpointing
    SAVE_EVERY = 10  # Save checkpoint every N epochs
    PATIENCE = 15  # Early stopping patience
    
    def __init__(self):
        # Create directories if they don't exist
        self.DATA_DIR.mkdir(exist_ok=True)
        self.MODELS_DIR.mkdir(exist_ok=True)
        self.OUTPUTS_DIR.mkdir(exist_ok=True)

config = Config()
print("Configuration initialized successfully")
print(f"  MIN_DF={config.MIN_DF}, MAX_DF={config.MAX_DF}, EMBEDDING_DIM={config.EMBEDDING_DIM}")


Configuration initialized successfully
  MIN_DF=10, MAX_DF=0.7, EMBEDDING_DIM=300


In [3]:
# Device configuration (for DETM model training - not for embeddings)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for DETM training: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print()

Using device for DETM training: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
CUDA Version: 12.8
Available GPU memory: 8.59 GB



---
## 3. Data Loading and Preprocessing

### 3.1 Download and Load UN General Debates Dataset

In [4]:
# check if data is already downloaded
data_file = config.DATA_DIR / 'un-general-debates.csv'
if data_file.exists():
    print(f"✓ Data file already exists at {data_file}")
else:
    !kaggle datasets download -d 'unitednations/un-general-debates' -p ../data --unzip

✓ Data file already exists at ../data/un-general-debates.csv


In [5]:
def load_and_explore_data():
    """Load and perform initial exploration of UN debates data"""
    
    # Load dataset
    csv_path = config.DATA_DIR / 'un-general-debates.csv'
    df = pd.read_csv(csv_path)
    
    print("=" * 60)
    print("UN GENERAL DEBATES DATASET")
    print("=" * 60)
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    
    # Date range
    if 'year' in df.columns:
        print(f"\nYear range: {df['year'].min()} - {df['year'].max()}")
        print(f"Years in dataset: {sorted(df['year'].unique())}")
    
    # Countries
    if 'country' in df.columns:
        print(f"\nNumber of countries: {df['country'].nunique()}")
        print(f"\nTop 10 countries by number of speeches:")
        print(df['country'].value_counts().head(10))
    
    # Text statistics
    if 'text' in df.columns:
        df['text_length'] = df['text'].astype(str).str.split().str.len()
        print(f"\nText length statistics:")
        print(df['text_length'].describe())
        
        # Note: Visualizations are handled by TensorBoard during training
        # For data exploration, refer to the printed statistics above
    
    return df

# Load and explore
df_raw = load_and_explore_data()

UN GENERAL DEBATES DATASET

Dataset shape: (7507, 4)

Columns: ['session', 'year', 'country', 'text']

Data types:
session     int64
year        int64
country    object
text       object
dtype: object

Missing values:
session    0
year       0
country    0
text       0
dtype: int64

Year range: 1970 - 2015
Years in dataset: [np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), n

### 3.2 Data Preprocessor Class

In [ ]:
class DataPreprocessor:
    """
    Preprocesses UN General Debates corpus for DETM.

    Aligned with original DETM (Dieng et al. 2019) preprocessing:

    1. Paragraph splitting — each speech is split on single newlines (\\n).
       Each non-empty line becomes a separate document. Speeches with no newlines
       are kept as a single document.

    2. Text cleaning — lowercase, NLTK word_tokenize, alpha-only filter, stopword
       removal (NLTK English + minimal UN procedural words), length filter (>2 chars).
       NO lemmatization — the original does not lemmatize.

    3. Vocabulary building — built from the TRAINING split only (matching original DETM).
       Document-frequency gates (MIN_DF / MAX_DF) applied. Val/test words absent from
       the train vocab are silently dropped in their BoW vectors.
    """

    # Minimal UN-specific stopwords — purely procedural/formulaic phrases only.
    # Topically meaningful words (international, world, global, countries, government,
    # security, development, peace …) are intentionally NOT removed — they carry
    # topic discrimination signal (e.g. "global warming", "international trade").
    UN_STOPWORDS = {
        # UN organisational boilerplate (name / procedural roles)
        "united", "nations", "assembly", "secretary", "delegation",
        "representative", "distinguished", "plenary", "headquarter", "secretariat",
        # Purely formulaic diplomatic phrases
        "reaffirm", "reiterate", "congratulate", "behalf", "honour",
        "convey", "utilize", "hereby",
    }

    def __init__(self, config):
        self.config = config
        # Standard NLTK English stopwords + minimal UN procedural stopwords
        self.stopwords = set(stopwords.words('english')) | self.UN_STOPWORDS
        self.vocabulary = None
        self.word2idx = None
        self.idx2word = None

    # ------------------------------------------------------------------
    # Paragraph splitting
    # ------------------------------------------------------------------

    def split_into_paragraphs(self, text: str) -> List[str]:
        """
        Split a speech into lines on single newlines (\\n).

        Each non-empty line becomes a separate document, giving finer-grained
        temporal documents that match the original DETM paragraph granularity.
        Speeches with no newlines at all are kept as a single document.

        Returns a list of non-empty line strings.
        """
        paras = [p.strip() for p in text.split('.\n') if p.strip()]
        return paras if paras else [text.strip()]

    # ------------------------------------------------------------------
    # Text cleaning — no lemmatization
    # ------------------------------------------------------------------

    def clean_text(self, text: str) -> List[str]:
        """
        Clean and tokenize a single document (paragraph or full speech).

        Pipeline: lowercase → word_tokenize → alpha-only → stopword removal
                  → length filter (>2 chars).
        Lemmatization is intentionally omitted (original DETM does not lemmatize).
        """
        text = str(text).lower()
        tokens = word_tokenize(text)
        tokens = [
            token
            for token in tokens
            if token.isalpha()
            and len(token) > 1
            and token not in self.stopwords
        ]
        return tokens

    # ------------------------------------------------------------------
    # Vocabulary building — df gates only, no IDF cap
    # ------------------------------------------------------------------

    def build_vocabulary(self, documents: List[List[str]]) -> Dict:
        """
        Build vocabulary using document-frequency gates only.

        All words passing both gates are kept — no IDF ranking, no cap.
        Gate values from config:
            MIN_DF = 10   (original; catches domain-specific rare words)
            MAX_DF = 0.7  (original; keeps important high-frequency topic markers)
        """
        print("Building vocabulary...")

        word_freq = Counter()
        doc_freq = Counter()

        for doc in tqdm(documents, desc="Counting frequencies"):
            word_freq.update(doc)
            doc_freq.update(set(doc))

        num_docs = len(documents)
        min_df_count = self.config.MIN_DF
        max_df_count = int(self.config.MAX_DF * num_docs)

        # Keep ALL words that pass both df gates (sorted alphabetically for stability)
        valid_words = sorted([
            word for word, cnt in doc_freq.items()
            if min_df_count <= cnt <= max_df_count
        ])

        self.word2idx = {word: idx for idx, word in enumerate(valid_words)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocabulary = valid_words

        print(f"\nVocabulary size: {len(self.vocabulary)}")
        print(f"Total unique tokens before filtering: {len(word_freq)}")
        print(f"Tokens passing df gates [{min_df_count}, {max_df_count}]: {len(valid_words)}")

        return {
            'word2idx': self.word2idx,
            'idx2word': self.idx2word,
            'vocabulary': self.vocabulary,
            'word_freq': {w: word_freq[w] for w in self.vocabulary},
            'doc_freq': {w: doc_freq[w] for w in self.vocabulary},
        }

    # ------------------------------------------------------------------
    # BoW conversion
    # ------------------------------------------------------------------

    def doc_to_bow(self, tokens: List[str]) -> np.ndarray:
        """Convert tokenized document to bag-of-words vector."""
        bow = np.zeros(len(self.vocabulary), dtype=np.float32)
        for token in tokens:
            if token in self.word2idx:
                bow[self.word2idx[token]] += 1
        return bow

    # ------------------------------------------------------------------
    # Temporal indexing (unchanged)
    # ------------------------------------------------------------------

    def _create_temporal_index(self, df: pd.DataFrame, bow_matrix: np.ndarray) -> Dict:
        """
        Create temporal indexing structure for DETM.

        Maps documents to time steps and computes average BoW per time period
        for the LSTM temporal baseline encoder.
        """
        if 'year' not in df.columns:
            print("Warning: No 'year' column found. Creating single time step.")
            return {
                'time_steps': [0],
                'doc_to_time': np.zeros(len(df), dtype=np.int64),
                'time_to_docs': {0: list(range(len(df)))},
                'avg_bow_per_time': np.asarray(bow_matrix.mean(axis=0)),  # shape (1, V) — works for both dense and sparse
                'num_time_steps': 1,
            }

        time_steps = sorted(df['year'].unique())
        time_to_idx = {year: idx for idx, year in enumerate(time_steps)}
        doc_to_time = np.array([time_to_idx[year] for year in df['year']], dtype=np.int64)

        time_to_docs = {t_idx: [] for t_idx in range(len(time_steps))}
        for doc_idx, t_idx in enumerate(doc_to_time):
            time_to_docs[t_idx].append(doc_idx)

        avg_bow_per_time = np.zeros((len(time_steps), bow_matrix.shape[1]), dtype=np.float32)
        for t_idx, doc_indices in time_to_docs.items():
            if len(doc_indices) > 0:
                time_docs = bow_matrix[doc_indices]  # CSR submatrix
                # Average raw counts: longer docs contribute proportionally more.
                # np.asarray flattens the (1, V) numpy.matrix that sparse .mean() returns.
                avg_bow_per_time[t_idx] = np.asarray(time_docs.mean(axis=0)).ravel()

        print(f"\n{'='*60}")
        print("TEMPORAL STRUCTURE")
        print(f"{'='*60}")
        print(f"Number of time steps (T): {len(time_steps)}")
        print(f"Time range: {time_steps[0]} - {time_steps[-1]}")
        print(f"Average documents per time step: {len(df) / len(time_steps):.1f}")
        print(f"Min docs in time step: {min(len(docs) for docs in time_to_docs.values())}")
        print(f"Max docs in time step: {max(len(docs) for docs in time_to_docs.values())}")

        return {
            'time_steps': time_steps,
            'doc_to_time': doc_to_time,
            'time_to_docs': time_to_docs,
            'avg_bow_per_time': avg_bow_per_time,
            'num_time_steps': len(time_steps),
        }

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------

    def preprocess_corpus(self, df: pd.DataFrame) -> Dict:
        """
        Complete preprocessing pipeline.

        Step 1 — Paragraph split: each speech → lines on \\n.
                  Speeches with no newlines are kept as one document.
                  Each line becomes an independent document (same year/country).
        Step 2 — Tokenize + clean each line/speech.
        Step 3 — Length filter on cleaned tokens (≥ MIN_DOC_LENGTH).
        Step 4 — Build vocabulary from training docs only (80% split, seed=42).
        Step 5 — Convert ALL docs to BoW using train vocab; OOV words silently dropped.
        Step 6 — Sort by year (temporal order).
        Step 7 — Build temporal index.

        Returns:
            Dictionary with:
            - bow_matrix:      Document-term matrix (N × V, float32)
            - tokens_list:     List of tokenized paragraph-documents
            - vocabulary_info: vocab / word2idx / word_freq / doc_freq
            - metadata:        Filtered DataFrame (one row per paragraph)
            - temporal_info:   Temporal structure for DETM LSTM
        """
        print("\n" + "=" * 60)
        print("PREPROCESSING CORPUS")
        print("=" * 60)
        print(f"Stopwords: {len(self.stopwords)} total "
              f"({len(set(stopwords.words('english')))} NLTK + "
              f"{len(self.UN_STOPWORDS)} UN-specific)")

        # Step 1: Split each speech into paragraphs → expanded corpus
        print(f"\nStep 1 — Splitting {len(df):,} speeches into paragraphs...")
        expanded_rows = []
        n_kept_whole = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Splitting paragraphs"):
            paras = self.split_into_paragraphs(str(row['text']))
            if len(paras) == 1:
                n_kept_whole += 1
            for para in paras:
                new_row = row.to_dict()
                new_row['text'] = para
                expanded_rows.append(new_row)
        df_expanded = pd.DataFrame(expanded_rows).reset_index(drop=True)
        print(f"  Speeches → paragraph-docs: {len(df):,} → {len(df_expanded):,}")
        print(f"  Speeches with no \\n kept as 1 doc (no newlines at all): {n_kept_whole:,} ({n_kept_whole/len(df)*100:.1f}%)")

        # Step 2: Tokenize + clean all paragraph-documents (no length filter yet)
        print("\nStep 2 — Tokenizing and cleaning paragraphs...")
        tokens_list_raw = []
        for text in tqdm(df_expanded['text'], desc="Tokenizing"):
            tokens_list_raw.append(self.clean_text(text))

        # Step 3: Apply length filter on cleaned tokens
        min_len = self.config.MIN_DOC_LENGTH
        valid_mask = [len(t) >= min_len for t in tokens_list_raw]
        df_filtered = df_expanded[valid_mask].reset_index(drop=True)
        tokens_list = [t for t, keep in zip(tokens_list_raw, valid_mask) if keep]
        print(f"  Documents after length filter (≥{min_len} tokens): {len(df_filtered):,} "
              f"(removed {sum(1 for v in valid_mask if not v):,})")

        # Step 4: Build vocabulary from training docs only (matching original DETM).
        # Replicate the same permutation used in create_dataloaders (seed=42, 80% train)
        # so that val/test-only words are excluded from the vocabulary.
        # Val/test doc_to_bow naturally drops any word not in word2idx (no special handling).
        _n_all = len(tokens_list)
        _train_perm = np.random.default_rng(42).permutation(_n_all)
        _n_vocab_train = int(_n_all * 0.85)
        train_tokens_for_vocab = [tokens_list[i] for i in _train_perm[:_n_vocab_train]]
        print(f"  Building vocabulary from {_n_vocab_train:,} training docs "
              f"({_n_vocab_train/_n_all*100:.0f}% of {_n_all:,} total)")
        vocab_info = self.build_vocabulary(train_tokens_for_vocab)

        # Step 5: Build sparse CSR BoW matrix — avoids ~18 GB dense allocation
        # that arises from sentence-level splitting (450K+ docs × 10K vocab × 4 bytes).
        # scipy.sparse CSR uses ~100× less RAM (only non-zero entries stored).
        print("\nStep 5 — Building sparse BoW (CSR format)...")
        V = len(self.vocabulary)
        _rows, _cols, _vals = [], [], []
        valid_indices = []
        row_ptr = 0

        for idx, tokens in enumerate(tqdm(tokens_list, desc="Creating sparse BoW")):
            token_counts = {}
            for token in tokens:
                if token in self.word2idx:
                    j = self.word2idx[token]
                    token_counts[j] = token_counts.get(j, 0) + 1
            if token_counts:
                for j, v in token_counts.items():
                    _rows.append(row_ptr)
                    _cols.append(j)
                    _vals.append(float(v))
                row_ptr += 1
                valid_indices.append(idx)

        bow_matrix = sparse.csr_matrix(
            (_vals, (_rows, _cols)), shape=(row_ptr, V), dtype=np.float32
        )
        df_filtered = df_filtered.iloc[valid_indices].reset_index(drop=True)
        tokens_list = [tokens_list[i] for i in valid_indices]

        nnz = bow_matrix.nnz
        dense_gb = row_ptr * V * 4 / 1e9
        sparse_mb = (nnz * 4 + nnz * 4 + (row_ptr + 1) * 4) / 1e6  # data+col+indptr
        print(f"\n  Final documents: {row_ptr:,}")
        print(f"  Vocabulary size: {V:,}")
        print(f"  Avg vocab tokens per document: {bow_matrix.sum(axis=1).mean():.1f}")
        print(f"  Sparsity: {(1 - nnz / (row_ptr * V)) * 100:.1f}%")
        print(f"  RAM: sparse ~{sparse_mb:.0f} MB  vs  dense ~{dense_gb:.1f} GB")

        # Step 6: Sort by temporal order
        if 'year' in df_filtered.columns:
            sort_idx = df_filtered['year'].argsort(kind='stable')
            bow_matrix = bow_matrix[sort_idx]
            df_filtered = df_filtered.iloc[sort_idx].reset_index(drop=True)
            tokens_list = [tokens_list[i] for i in sort_idx]
            print(f"  Documents sorted by year: {df_filtered['year'].min()} → {df_filtered['year'].max()}")

        # Step 7: Build temporal index
        temporal_info = self._create_temporal_index(df_filtered, bow_matrix)

        processed_data = {
            'bow_matrix': bow_matrix,
            'tokens_list': tokens_list,
            'vocabulary_info': vocab_info,
            'metadata': df_filtered,
            'num_docs': bow_matrix.shape[0],
            'vocab_size': len(self.vocabulary),
            'temporal_info': temporal_info,
        }

        save_path = self.config.DATA_DIR / 'preprocessed_data.pkl'
        with open(save_path, 'wb') as f:
            pickle.dump(processed_data, f)
        print(f"\nPreprocessed data saved to: {save_path}")

        return processed_data


print("DataPreprocessor class defined.")


DataPreprocessor class defined.


In [7]:
# Usage example:
preprocessor = DataPreprocessor(config)
processed_data = preprocessor.preprocess_corpus(df_raw)


PREPROCESSING CORPUS
Stopwords: 216 total (198 NLTK + 18 UN-specific)

Step 1 — Splitting 7,507 speeches into paragraphs...


Splitting paragraphs:   0%|          | 0/7507 [00:00<?, ?it/s]

  Speeches → paragraph-docs: 7,507 → 1,377,801
  Speeches with no \n kept as 1 doc (no newlines at all): 0 (0.0%)

Step 2 — Tokenizing and cleaning paragraphs...


Tokenizing:   0%|          | 0/1377801 [00:00<?, ?it/s]

  Documents after length filter (≥5 tokens): 592,355 (removed 785,446)
  Building vocabulary from 473,884 training docs (80% of 592,355 total)
Building vocabulary...


Counting frequencies:   0%|          | 0/473884 [00:00<?, ?it/s]


Vocabulary size: 15585
Total unique tokens before filtering: 41922
Tokens passing df gates [10, 331718]: 15585

Step 5 — Building sparse BoW (CSR format)...


Creating sparse BoW:   0%|          | 0/592355 [00:00<?, ?it/s]


  Final documents: 592,345
  Vocabulary size: 15,585
  Avg vocab tokens per document: 13.2
  Sparsity: 99.9%
  RAM: sparse ~60 MB  vs  dense ~36.9 GB
  Documents sorted by year: 1970 → 2015

TEMPORAL STRUCTURE
Number of time steps (T): 46
Time range: 1970 - 2015
Average documents per time step: 12877.1
Min docs in time step: 3208
Max docs in time step: 33643

Preprocessed data saved to: ../data/preprocessed_data.pkl


### 3.3 Vocabulary Quality Diagnostic

Verify the vocabulary looks correct before training:
- Top words (highest IDF) should be specific domain terms, **not** UN boilerplate
- Known boilerplate words ("united", "nations", "assembly"…) should be absent
- Lemmatization should have collapsed morphological variants

In [8]:
# ── Vocabulary Quality Diagnostic ────────────────────────────────────────────
vocab = preprocessor.vocabulary
vocab_info = processed_data['vocabulary_info']
word_freq = vocab_info['word_freq']
doc_freq = vocab_info['doc_freq']
n_docs = processed_data['num_docs']

print('=' * 70)
print('VOCABULARY QUALITY CHECK')
print('=' * 70)
print(f'Vocabulary size: {len(vocab):,}')
print(f'Total documents (paragraphs): {n_docs:,}')

# Vocabulary is alphabetically sorted; show words ranked by document frequency
vocab_by_df_desc = sorted(vocab, key=lambda w: doc_freq[w], reverse=True)
vocab_by_df_asc  = sorted(vocab, key=lambda w: doc_freq[w])

print('\n--- Top 50 most frequent words (highest doc freq — should include topic markers) ---')
for i, w in enumerate(vocab_by_df_desc[:50]):
    print(f'  {i+1:>3}. {w:<22} df={doc_freq[w]:>6} ({doc_freq[w]/n_docs*100:.1f}%)')

print('\n--- Bottom 20 words (lowest doc freq — near MIN_DF gate) ---')
for i, w in enumerate(vocab_by_df_asc[:20]):
    print(f'  {i+1:>3}. {w:<22} df={doc_freq[w]:>6} ({doc_freq[w]/n_docs*100:.2f}%)')

print('\n--- Procedural stopwords: should NOT be in vocabulary ---')
procedural_check = [
    'united', 'nations', 'assembly', 'secretary', 'delegation',
    'representative', 'distinguished', 'plenary',
    'reaffirm', 'reiterate', 'congratulate', 'hereby',
]
found  = [w for w in procedural_check if w in preprocessor.word2idx]
absent = [w for w in procedural_check if w not in preprocessor.word2idx]
print(f'  ✓ Correctly removed ({len(absent)}): {absent}')
if found:
    print(f'  ⚠ Still present ({len(found)}): {found}')
else:
    print('  ✓ All procedural stopwords successfully filtered!')

print('\n--- Topic-signal words: SHOULD be in vocabulary (no longer over-filtered) ---')
topic_signal_words = [
    'international', 'global', 'world', 'security', 'development', 'peace',
    'government', 'countries', 'economic', 'human', 'rights',
    'climate', 'nuclear', 'trade', 'conflict', 'environment',
]
in_vocab  = [w for w in topic_signal_words if w in preprocessor.word2idx]
not_vocab = [w for w in topic_signal_words if w not in preprocessor.word2idx]
print(f'  ✓ In vocabulary ({len(in_vocab)}): {in_vocab}')
if not_vocab:
    print(f'  ⚠ Missing (may be below MIN_DF={preprocessor.config.MIN_DF} or above MAX_DF={preprocessor.config.MAX_DF}): {not_vocab}')

print('\n--- Morphological variants (no lemmatization — all forms retained) ---')
morph_groups = [
    ('develop', 'development', 'developing', 'developed'),
    ('terror', 'terrorism', 'terrorist'),
    ('economic', 'economy', 'economies'),
    ('nuclear',),
    ('peace', 'peaceful', 'peacefully'),
]
for group in morph_groups:
    present_forms = [w for w in group if w in preprocessor.word2idx]
    print(f'  {group[0]:>15}: forms in vocab → {present_forms}')

print('\n--- 20 random vocabulary words (sanity check) ---')
rng = np.random.default_rng(42)
idx_sample = rng.choice(len(vocab), size=20, replace=False)
print(' ', ', '.join([vocab[int(i)] for i in sorted(idx_sample)]))


VOCABULARY QUALITY CHECK
Vocabulary size: 15,585
Total documents (paragraphs): 592,345

--- Top 50 most frequent words (highest doc freq — should include topic markers) ---
    1. international          df= 50921 (8.6%)
    2. world                  df= 46029 (7.8%)
    3. peace                  df= 37883 (6.4%)
    4. countries              df= 37480 (6.3%)
    5. states                 df= 30418 (5.1%)
    6. people                 df= 29422 (5.0%)
    7. must                   df= 29235 (4.9%)
    8. also                   df= 28958 (4.9%)
    9. security               df= 28363 (4.8%)
   10. economic               df= 27003 (4.6%)
   11. new                    df= 26279 (4.4%)
   12. development            df= 23960 (4.0%)
   13. would                  df= 20814 (3.5%)
   14. country                df= 20170 (3.4%)
   15. human                  df= 20046 (3.4%)
   16. efforts                df= 19332 (3.3%)
   17. political              df= 19208 (3.2%)
   18. one                  

---
## 4. Word Embeddings Generation

### 4.1 Word2Vec Embedding Generator

**Why Word2Vec instead of SentenceTransformer?**

DETM computes topic-word distributions as `β_k = softmax(α_k · ρ^T)`.
For topics to be distinct, word embeddings ρ must be **angularly spread** across the embedding space so that different α_k vectors point towards different clusters.

- **SentenceTransformer (all-MiniLM-L6-v2)** encodes single words using 6 transformer layers + mean-pool. Single-word outputs cluster in a narrow angular cone (avg pairwise cosine ~0.65–0.80 for unrelated words) because the representation is dominated by positional/syntactic context frames, not distributional co-occurrence geometry. This makes all β_k nearly identical regardless of α_k → topic collapse.

- **Word2Vec skip-gram** directly optimises the inner-product objective: *given word w, predict context words*. The resulting geometry is specifically designed for inner-product factorisation — exactly what DETM's `α · ρ^T` assumes. Embeddings are much more uniformly distributed over the sphere (avg pairwise cosine ~0.05–0.20).

Training on the preprocessing corpus ensures the embedding space reflects the actual vocabulary distribution of UN speeches.


In [9]:
class EmbeddingGenerator:
    """
    Generates word embeddings using Word2Vec skip-gram trained on the corpus.
    
    Replaces the previous SentenceTransformer approach. Rationale:
    
    DETM's decoder computes β_k = softmax(α_k · ρ^T).  After L2-normalising ρ,
    this becomes β_k,w ∝ exp(||α_k|| · cos(α_k, ρ_w)).  For topics to be distinct,
    word embeddings must be spread across the angular space — a property that
    Word2Vec skip-gram provides directly (its training objective optimises exactly
    the inner-product factorisation that DETM needs).
    
    SentenceTransformer single-word embeddings cluster in a narrow cone
    (avg pairwise cosine ~0.65–0.80) because they capture syntactic/positional
    context, not co-occurrence geometry.  This causes posterior collapse.
    
    Architecture (matching Dieng et al. 2019):
        - Algorithm: skip-gram  (sg=1)
        - Vector size: 300  (config.EMBEDDING_DIM)
        - Window: 5         (config.W2V_WINDOW)
        - min_count: same as config.MIN_DF (consistent with vocab gates)
        - Epochs: 10        (config.W2V_EPOCHS)
    
    Caching: trained Word2Vec model is saved to DATA_DIR/word2vec.model and
    reloaded on subsequent runs, avoiding unnecessary retraining.
    Similarly, the extracted embedding matrix is saved to DATA_DIR/word_embeddings.npy.
    
    OOV handling: words in the vocabulary that are not in the W2V model (extremely
    rare since both use the same cleaned corpus) receive a small random unit vector.
    """

    W2V_FILENAME = 'word2vec.model'
    EMB_FILENAME  = 'word_embeddings.npy'

    def __init__(self, config):
        self.config = config
        self.model = None
        self.embedding_dim = config.EMBEDDING_DIM

    def _w2v_path(self) -> Path:
        return self.config.DATA_DIR / self.W2V_FILENAME

    def _emb_path(self) -> Path:
        return self.config.DATA_DIR / self.EMB_FILENAME

    def train_word2vec(self, tokens_list: List[List[str]]) -> 'Word2Vec':
        """
        Train Word2Vec skip-gram on the tokenized corpus and save to disk.
        
        Args:
            tokens_list: List of tokenized documents (already cleaned + lemmatized)
        
        Returns:
            Trained gensim Word2Vec model
        """
        print("\n" + "=" * 60)
        print("TRAINING WORD2VEC SKIP-GRAM")
        print("=" * 60)
        print(f"Corpus size: {len(tokens_list):,} documents")
        print(f"Total tokens: {sum(len(t) for t in tokens_list):,}")
        print(f"Parameters: vector_size={self.embedding_dim}, "
              f"window={self.config.W2V_WINDOW}, "
              f"min_count={self.config.MIN_DF}, "
              f"sg=1 (skip-gram), "
              f"epochs={self.config.W2V_EPOCHS}")

        np.random.seed(SEED)
        self.model = Word2Vec(
            sentences=tokens_list,
            vector_size=self.embedding_dim,
            window=self.config.W2V_WINDOW,
            min_count=self.config.MIN_DF,  # consistent with vocab frequency gate
            workers=4,
            sg=1,    # skip-gram (vs. CBOW)
            seed=SEED,
            epochs=self.config.W2V_EPOCHS,
        )
        save_path = self._w2v_path()
        self.model.save(str(save_path))
        print(f"✓ Word2Vec trained — vocabulary: {len(self.model.wv):,} words")
        print(f"✓ Model saved to: {save_path}")
        return self.model

    def load_word2vec(self) -> 'Word2Vec':
        """Load a previously saved Word2Vec model from disk."""
        path = self._w2v_path()
        self.model = Word2Vec.load(str(path))
        print(f"✓ Word2Vec loaded from cache: {path} "
              f"({len(self.model.wv):,} words)")
        return self.model

    def generate_vocabulary_embeddings(
        self,
        vocabulary: List[str],
        tokens_list: Optional[List[List[str]]] = None,
        force_retrain: bool = False,
    ) -> np.ndarray:
        """
        Extract vocabulary embeddings, using cached files when available.
        
        Cache behaviour:
          - If DATA_DIR/word2vec.model exists AND force_retrain=False → load cached W2V.
          - Otherwise train from scratch (tokens_list required in that case).
          - Extracted embedding matrix is always saved to DATA_DIR/word_embeddings.npy.
        
        Args:
            vocabulary:    Ordered vocabulary list (from DataPreprocessor)
            tokens_list:   Tokenized corpus — required only when training from scratch
            force_retrain: If True, ignore any cached model and retrain
        
        Returns:
            Embedding matrix (vocab_size, embedding_dim), float32
        """
        w2v_path = self._w2v_path()

        if not force_retrain and w2v_path.exists():
            print(f"✓ Cached Word2Vec model found — loading (skip training).")
            print(f"  To retrain, call with force_retrain=True or delete {w2v_path}")
            self.load_word2vec()
        else:
            if tokens_list is None:
                raise ValueError(
                    "tokens_list is required to train Word2Vec. "
                    f"No cached model found at {w2v_path}."
                )
            self.train_word2vec(tokens_list)

        print("\n" + "=" * 60)
        print("EXTRACTING VOCABULARY EMBEDDINGS")
        print("=" * 60)
        print(f"Vocabulary size: {len(vocabulary):,}")

        embeddings = np.zeros((len(vocabulary), self.embedding_dim), dtype=np.float32)
        oov_words = []

        for idx, word in enumerate(vocabulary):
            if word in self.model.wv:
                embeddings[idx] = self.model.wv[word]
            else:
                # OOV: random unit vector (extremely rare — same min_count applied)
                oov_vec = np.random.randn(self.embedding_dim).astype(np.float32)
                embeddings[idx] = oov_vec / (np.linalg.norm(oov_vec) + 1e-10)
                oov_words.append(word)

        if oov_words:
            print(f"⚠ OOV words ({len(oov_words)}): {oov_words[:10]}"
                  f"{'...' if len(oov_words) > 10 else ''}")
        else:
            print("✓ All vocabulary words found in Word2Vec model (0 OOV)")

        print(f"\nEmbedding matrix shape: {embeddings.shape}")
        print(f"Embedding statistics (before L2-norm):")
        print(f"  Mean: {embeddings.mean():.4f}")
        print(f"  Std:  {embeddings.std():.4f}")
        print(f"  Min:  {embeddings.min():.4f}")
        print(f"  Max:  {embeddings.max():.4f}")

        # Save raw embeddings (DETMDecoder will L2-normalise them at init)
        emb_path = self._emb_path()
        np.save(emb_path, embeddings)
        print(f"✓ Embeddings saved to: {emb_path}")

        return embeddings


print("EmbeddingGenerator class defined (Word2Vec skip-gram, with caching).")


EmbeddingGenerator class defined (Word2Vec skip-gram, with caching).


In [10]:
# Train (or load cached) Word2Vec skip-gram and extract vocabulary embeddings.
# Pass tokens_list so training can run on first execution;
# subsequent runs load the saved model from data/word2vec.model automatically.
embedding_gen = EmbeddingGenerator(config)
word_embeddings = embedding_gen.generate_vocabulary_embeddings(
    preprocessor.vocabulary,
    tokens_list=processed_data['tokens_list'],  # ignored if cached model exists
)


✓ Cached Word2Vec model found — loading (skip training).
  To retrain, call with force_retrain=True or delete ../data/word2vec.model
✓ Word2Vec loaded from cache: ../data/word2vec.model (11,084 words)

EXTRACTING VOCABULARY EMBEDDINGS
Vocabulary size: 15,585
⚠ OOV words (4768): ['abandons', 'abate', 'abatement', 'abdallah', 'aberrant', 'aberrations', 'abetting', 'abeyance', 'abhorred', 'abhors']...

Embedding matrix shape: (15585, 300)
Embedding statistics (before L2-norm):
  Mean: -0.0043
  Std:  0.1898
  Min:  -1.4018
  Max:  1.3503
✓ Embeddings saved to: ../data/word_embeddings.npy


### 4.2 Embedding Quality Diagnostic

Key checks before training:
- **Angular spread**: average pairwise cosine similarity should be low (~0.05–0.20 for Word2Vec vs ~0.65–0.80 for SentenceTransformer). Low spread = topics can differentiate.
- **Nearest neighbours**: semantically related words should cluster together.
- **Post-L2-norm geometry**: since `DETMDecoder` L2-normalises ρ, we check the normalised dot-product distribution.

In [11]:
# ── Embedding Quality Diagnostic ─────────────────────────────────────────────
rng = np.random.default_rng(42)
n_sample = 500  # number of random pairs for cosine similarity estimate

# L2-normalise once (mirrors what DETMDecoder does at init)
norms = np.linalg.norm(word_embeddings, axis=1, keepdims=True) + 1e-10
emb_norm = word_embeddings / norms

print('=' * 70)
print('EMBEDDING QUALITY DIAGNOSTIC  (Word2Vec skip-gram)')
print('=' * 70)

# 1. Angular spread: sample random pairs and compute cosine similarity
idx_a = rng.integers(0, len(preprocessor.vocabulary), size=n_sample)
idx_b = rng.integers(0, len(preprocessor.vocabulary), size=n_sample)
# avoid self-pairs
mask = idx_a != idx_b
idx_a, idx_b = idx_a[mask], idx_b[mask]
cos_sims = (emb_norm[idx_a] * emb_norm[idx_b]).sum(axis=1)

print(f'\nPairwise cosine similarity ({len(cos_sims)} random pairs):')
print(f'  Mean:   {cos_sims.mean():.4f}   (Word2Vec target: 0.05–0.20)')
print(f'  Median: {np.median(cos_sims):.4f}')
print(f'  Std:    {cos_sims.std():.4f}')
print(f'  p10:    {np.percentile(cos_sims, 10):.4f}')
print(f'  p90:    {np.percentile(cos_sims, 90):.4f}')
if cos_sims.mean() < 0.3:
    print('  ✓ Embeddings are well-spread (low pairwise similarity)')
else:
    print('  ⚠ Embeddings may be clustered — expected <0.3 for Word2Vec')

# 2. Nearest-neighbour sanity checks
# No lemmatization — probe with actual surface forms that appear in the corpus
probe_words = ['nuclear', 'trade', 'climate', 'terrorism', 'development',
               'conflict', 'poverty', 'democracy', 'disarmament', 'environment',
               'security', 'international', 'global']
print('\nNearest neighbours (top-5, by cosine similarity):')
w2v = embedding_gen.model
for word in probe_words:
    if word in w2v.wv:
        neighbours = w2v.wv.most_similar(word, topn=5)
        nn_str = ', '.join([f"{w}({s:.2f})" for w, s in neighbours])
        print(f'  {word:<16} → {nn_str}')
    else:
        print(f'  {word:<16} → not in W2V vocabulary')

# 3. Embedding norm distribution
raw_norms = np.linalg.norm(word_embeddings, axis=1)
print(f'\nRaw embedding L2-norm distribution:')
print(f'  Mean: {raw_norms.mean():.4f}  Std: {raw_norms.std():.4f}  '
      f'Min: {raw_norms.min():.4f}  Max: {raw_norms.max():.4f}')
print(f'  (After L2-norm in DETMDecoder all norms → 1.0)')


EMBEDDING QUALITY DIAGNOSTIC  (Word2Vec skip-gram)

Pairwise cosine similarity (500 random pairs):
  Mean:   0.0406   (Word2Vec target: 0.05–0.20)
  Median: 0.0385
  Std:    0.0808
  p10:    -0.0601
  p90:    0.1450
  ✓ Embeddings are well-spread (low pairwise similarity)

Nearest neighbours (top-5, by cosine similarity):
  nuclear          → weapon(0.61), atomic(0.59), npt(0.58), testing(0.56), nonproliferation(0.54)
  trade            → trading(0.59), gatt(0.56), wto(0.54), tariff(0.53), liberalization(0.51)
  climate          → change(0.61), warming(0.57), kyoto(0.49), environment(0.49), unfccc(0.49)
  terrorism        → terrorist(0.60), extremism(0.55), fight(0.50), combating(0.50), terror(0.48)
  development      → sustainable(0.62), developmental(0.51), mdgs(0.50), goal(0.48), growth(0.47)
  conflict         → dispute(0.53), flare(0.51), flashpoint(0.50), strife(0.49), erupting(0.48)
  poverty          → hunger(0.73), illiteracy(0.66), underdevelopment(0.62), disease(0.59), malnu

---
## 5. DETM Model Architecture

### 5.1 Document Topic Encoder (Amortized Inference with Logistic-Normal)

In [12]:
class DocumentTopicEncoder(nn.Module):
    """
    Amortized Variational Inference for Document Topic Proportions (θ_d).
    
    Uses MLP-based amortized inference conditioned on both document text and
    temporal baseline. Outputs logistic-normal distribution (softmax of Gaussian).
    
    Architecture:
        Inputs: 
            - w_d: Document BoW (V)
            - η_{t_d}: Temporal baseline for document's year (K)
        ↓
        Concatenate: [w_d, η_{t_d}] → (V + K)
        ↓
        MLP: 
            - Linear(V+K → 800) → ReLU → Dropout(0.1)
            - Linear(800 → 800) → ReLU → Dropout(0.1)
        ↓
        Two parallel outputs:
            - μ_θ: Linear(800 → K)
            - log σ²_θ: Linear(800 → K)
        ↓
        Reparameterization:
            - γ_d = μ_θ + ε ⊙ exp(0.5 · log σ²_θ)
            - θ_d = softmax(γ_d)  # logistic-normal transformation
    """
    
    def __init__(
        self,
        vocab_size: int,
        num_topics: int,
        hidden_dim: int = 800,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.num_topics = num_topics
        self.hidden_dim = hidden_dim
        
        # Input is concatenated [BoW (V), eta (K)]
        input_dim = vocab_size + num_topics
        
        # MLP with 2 hidden layers (no dropout — matches original enc_drop=0.0)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Output layers for mean and log variance
        self.fc_mu = nn.Linear(hidden_dim, num_topics)
        self.fc_logvar = nn.Linear(hidden_dim, num_topics)
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        Reparameterization trick: γ = μ + σ * ε where ε ~ N(0, I).
        Returns μ deterministically at eval time (no noise during validation/inference).
        """
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def forward(
        self,
        bow: torch.Tensor,
        eta: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass through document topic encoder.
        
        Args:
            bow: Bag-of-words tensor (batch_size, vocab_size)
            eta: Temporal baseline (batch_size, num_topics)
        
        Returns:
            theta: Topic proportions on simplex (batch_size, num_topics)
            mu: Mean of pre-softmax Gaussian (batch_size, num_topics)
            logvar: Log variance of pre-softmax Gaussian (batch_size, num_topics)
        """
        # Normalize BoW (simple L1-norm, matching original — no BatchNorm)
        bow_norm = bow / (bow.sum(dim=1, keepdim=True) + 1e-10)
        
        # Concatenate BoW and temporal baseline
        combined = torch.cat([bow_norm, eta], dim=1)  # (batch_size, V + K)
        
        # Encode
        hidden = self.encoder(combined)  # (batch_size, 800)
        
        # Compute variational parameters
        mu = self.fc_mu(hidden)  # (batch_size, K)
        logvar = self.fc_logvar(hidden)  # (batch_size, K)
        
        # Sample from Gaussian
        gamma = self.reparameterize(mu, logvar)  # (batch_size, K)
        
        # Apply logistic-normal transformation: θ = softmax(γ)
        theta = F.softmax(gamma, dim=-1)  # (batch_size, K) - now on simplex
        
        return theta, mu, logvar


### 
5.2 Decoder (Generative Model with External Topic Embeddings)

In [13]:


class DETMDecoder(nn.Module):
    """
    Generative network for DETM.
    
    Models word generation given topic proportions and topic embeddings:
        p(w | θ, α, ρ) where:
        - θ: document topic proportions (batch_size, num_topics)
        - α: topic embeddings (num_topics, embedding_dim) or (batch_size, num_topics, embedding_dim)
        - ρ: word embeddings (vocab_size, embedding_dim)  ← raw W2V, trainable
    
    Topic-word distribution: β_k = softmax(α_k · ρ^T)
    The raw W2V inner-product geometry is preserved (no L2-normalization).
    Word embeddings are trainable by default, matching the original implementation.
    
    Word probability: p(w) = θ^T · β
    
    Note: Topic embeddings α are NOT stored here - they are passed as input.
          This allows for time-varying α^(t) from mean-field parameters.
    """
    
    def __init__(
        self,
        num_topics: int,
        vocab_size: int,
        embedding_dim: int,
        word_embeddings: torch.Tensor,
        train_word_embeddings: bool = False
    ):
        super().__init__()
        
        self.num_topics = num_topics
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # Store raw word embeddings (NO L2-normalization — matches original).
        # L2-norm compresses logit dynamic range and with small α init causes
        # β ≈ uniform → posterior collapse.  Raw W2V norms encode word importance.
        self.word_embeddings = nn.Parameter(
            word_embeddings.clone(),
            requires_grad=train_word_embeddings
        )
    
    def get_beta(self, topic_embeddings: torch.Tensor) -> torch.Tensor:
        """
        Compute topic-word distribution matrix β.
        
        β[k, w] = p(w | topic=k) = softmax_w(α_k · ρ_w)
        
        Args:
            topic_embeddings: Topic embeddings α, shape:
                - (num_topics, embedding_dim) for shared α across batch
                - (batch_size, num_topics, embedding_dim) for time-varying α
        
        Returns:
            beta: Topic-word distribution of shape (num_topics, vocab_size)
                  or (batch_size, num_topics, vocab_size)
        """
        if topic_embeddings.dim() == 2:
            # Shared topic embeddings: (K, L)
            # Compute logits: α · ρ^T = (K, L) × (L, V) = (K, V)
            logits = torch.mm(topic_embeddings, self.word_embeddings.t())
            # Apply softmax over vocabulary dimension
            beta = F.softmax(logits, dim=-1)  # (K, V)
            
        elif topic_embeddings.dim() == 3:
            # Time-varying topic embeddings: (B, K, L)
            # Compute logits: α · ρ^T = (B, K, L) × (L, V) = (B, K, V)
            logits = torch.matmul(topic_embeddings, self.word_embeddings.t())
            # Apply softmax over vocabulary dimension
            beta = F.softmax(logits, dim=-1)  # (B, K, V)
        else:
            raise ValueError(f"Invalid topic_embeddings shape: {topic_embeddings.shape}")
        
        return beta
    
    def forward(
        self,
        theta: torch.Tensor,
        topic_embeddings: torch.Tensor
    ) -> torch.Tensor:
        """
        Generate word distribution for documents.
        
        Args:
            theta: Document topic proportions (batch_size, num_topics)
                   Already on simplex (from logistic-normal)
            topic_embeddings: Topic embeddings α, shape:
                - (num_topics, embedding_dim) for static/shared
                - (batch_size, num_topics, embedding_dim) for time-varying
        
        Returns:
            word_dist: Word probability distribution (batch_size, vocab_size)
        """
        # Get topic-word distribution
        beta = self.get_beta(topic_embeddings)  # (K, V) or (B, K, V)
        
        # Compute document-word distribution
        # p(w | d) = θ^T · β
        
        if beta.dim() == 2:
            # Shared β: (K, V)
            # θ: (B, K), β: (K, V) → (B, K) × (K, V) = (B, V)
            word_dist = torch.mm(theta, beta)
        else:
            # Time-varying β: (B, K, V)
            # θ: (B, K, 1), β: (B, K, V) → (B, K, 1) × (B, K, V) = (B, V)
            word_dist = torch.bmm(theta.unsqueeze(1), beta).squeeze(1)
        
        return word_dist

print("DETMDecoder class defined.")


DETMDecoder class defined.


### 5.3 Temporal Baseline Encoder (LSTM-based Structured Inference)

In [14]:
class TemporalBaselineEncoder(nn.Module):
    """
    LSTM-based Structured Variational Inference for Global Temporal Baselines (η_t).
    
    This component models the overall topic popularity across the corpus for each
    time step using a structured variational family.
    
    Architecture:
        Input: w̃_t (V-dimensional average normalized BoW for year t)
        ↓
        Compression: Linear(V → compression_dim)
        ↓
        LSTM: 3 layers, 200 hidden units each
        ↓
        For each time step t:
            - Residual: Concat [LSTM_output_t (200), η_{t-1} (K)]
            - Two parallel Linear layers → μ_η (K), log σ²_η (K)
            - Sample: η_t = μ_η + ε ⊙ exp(0.5 · log σ²_η)
            - Compute KL(q(η_t) || p(η_t | η_{t-1})) using the FRESH η_{t-1} sample
    
    KL is computed *during* the forward pass so that η_{t-1} is still part of the
    live computation graph — this enables Monte Carlo gradients without graph reuse.
    
    Prior: p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)  where δ² = ETA_PRIOR_VARIANCE
    """
    
    def __init__(
        self,
        vocab_size: int,
        num_topics: int,
        compression_dim: int = 400,
        lstm_layers: int = 4,
        lstm_hidden: int = 400,
        delta_sq: float = 0.005  # δ²: prior variance for the random walk
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.num_topics = num_topics
        self.compression_dim = compression_dim
        self.lstm_layers = lstm_layers
        self.lstm_hidden = lstm_hidden
        self.delta_sq = delta_sq  # δ²: p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
        
        # Compression layer: V → 400
        self.compress = nn.Linear(vocab_size, compression_dim)
        
        # LSTM: sequential over time steps
        self.lstm = nn.LSTM(
            input_size=compression_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=0.0  # Original uses eta_dropout=0.0; dropout on η-LSTM destabilizes temporal prior
        )
        
        # Output layers: Input is [LSTM_output (400), η_{t-1} (K)] → 400 + K
        self.fc_mu = nn.Linear(lstm_hidden + num_topics, num_topics)
        self.fc_logvar = nn.Linear(lstm_hidden + num_topics, num_topics)
        
        # eta_timeline cached for inference use (get_document_topics)
        self.eta_timeline = None
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample N(mu, exp(logvar)) at train time; return mu at eval (no noise)."""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def forward(
        self,
        avg_bow_timeline: torch.Tensor,
        return_params: bool = False
    ) -> torch.Tensor:
        """
        Process entire timeline to generate global temporal baselines.
        
        Computes KL divergence *during* the loop so that η_{t-1} is a fresh
        sample in the current computation graph (Monte Carlo, matches paper).
        
        Prior: p(η_0) = N(0, δ²I),  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)
        
        KL(N(μ, σ²) || N(m, δ²)) = 0.5 * [σ²/δ² + (μ-m)²/δ² - 1 - log(σ²/δ²)]
        """
        T = avg_bow_timeline.shape[0]
        
        # Compress: (T, V) → (T, compression_dim)
        compressed = self.compress(avg_bow_timeline)
        
        # LSTM over full timeline
        lstm_input = compressed.unsqueeze(0)   # (1, T, compression_dim)
        lstm_output, _ = self.lstm(lstm_input) # (1, T, lstm_hidden)
        lstm_output = lstm_output.squeeze(0)   # (T, lstm_hidden)
        
        eta_mu_list = []
        eta_logvar_list = []
        eta_list = []
        kl_terms = []
        
        # η_prev starts at zero (used as the prior mean for t=0)
        eta_prev = torch.zeros(1, self.num_topics, device=avg_bow_timeline.device)
        
        for t in range(T):
            lstm_t = lstm_output[t:t+1]                         # (1, lstm_hidden)
            combined = torch.cat([lstm_t, eta_prev], dim=1)     # (1, lstm_hidden + K)
            
            mu_t     = self.fc_mu(combined)                     # (1, K)
            logvar_t = self.fc_logvar(combined)                 # (1, K)
            eta_t    = self.reparameterize(mu_t, logvar_t)      # (1, K)
            
            # KL(q(η_t) || p(η_t | η_prev)) = KL(N(μ_t, σ²_t) || N(η_prev, δ²I))
            # For t=0: η_prev = 0, so this is KL(N(μ_0, σ²_0) || N(0, I))
            var_t = torch.exp(logvar_t)
            if t == 0:
                # Prior is N(0, I) — NOT N(0, δ²·I)
                kl_t = 0.5 * torch.sum(
                    var_t + mu_t.pow(2) - 1.0 - logvar_t
                )
            else:
                # Prior is N(η_prev, δ²·I) — unchanged
                kl_t = 0.5 * torch.sum(
                    var_t / self.delta_sq
                    + (mu_t - eta_prev).pow(2) / self.delta_sq
                    - 1.0 - logvar_t + math.log(self.delta_sq)
                )
            kl_terms.append(kl_t)
            
            # eta_prev for the next step is the FRESH sample (still in graph)
            eta_prev = eta_t
            
            eta_mu_list.append(mu_t)
            eta_logvar_list.append(logvar_t)
            eta_list.append(eta_t)
        
        eta_mu_timeline    = torch.cat(eta_mu_list,    dim=0)  # (T, K)
        eta_logvar_timeline = torch.cat(eta_logvar_list, dim=0) # (T, K)
        eta_timeline       = torch.cat(eta_list,       dim=0)  # (T, K)
        
        # Cache eta_timeline for inference-time use (e.g. get_document_topics)
        self.eta_timeline = eta_timeline
        
        # kl_eta is a live tensor — gradients flow back to LSTM/fc_mu/fc_logvar
        kl_eta = sum(kl_terms)
        
        if return_params:
            return eta_timeline, eta_mu_timeline, eta_logvar_timeline, kl_eta
        return eta_timeline, kl_eta

print("TemporalBaselineEncoder class defined.")


TemporalBaselineEncoder class defined.


---
## 6. Loss Modules (Modular Design)

### 6.1 Reconstruction Loss

In [ ]:
class ReconstructionLoss(nn.Module):
    """
    Computes reconstruction loss for DETM.
    
    Measures how well the model reconstructs the observed word counts.
    Loss = -E_q(θ)[log p(w | θ, α, ρ)]
    """
    
    def __init__(self, loss_type: str = 'multinomial'):
        super().__init__()
        self.loss_type = loss_type
    
    def multinomial_loss(self, bow, word_dist):
        """
        Multinomial negative log-likelihood — per-document total (not per-word).

        Matches the original DETM (Dieng et al. 2019) which sums over vocabulary
        without dividing by document length. Length-normalization (÷ doc_length)
        converts NLL to per-word scale (~50× smaller than un-normalized KL terms),
        causing KL to dominate → all topics collapse to the prior.

        Returns: Per-document NLL (batch_size,)
        """
        word_dist = torch.clamp(word_dist, min=1e-10, max=1.0)
        nll = -(bow * torch.log(word_dist)).sum(dim=-1)  # per-doc total NLL, no length norm
        return nll
    
    def poisson_loss(self, bow, word_dist):
        """Poisson negative log-likelihood. Returns per-document NLL (batch_size,)."""
        doc_lengths = bow.sum(dim=-1, keepdim=True)
        rates = torch.clamp(word_dist * doc_lengths, min=1e-10)
        nll = (rates - bow * torch.log(rates)).sum(dim=-1)
        return nll
    
    def forward(self, bow, word_dist):
        """
        Return per-document NLL — shape (B,). Caller handles aggregation.

        Returning (B,) instead of a scalar mean lets DETM.forward() apply the
        corpus-scale coeff (.sum() * coeff) matching Dieng et al. 2019.
        """
        if self.loss_type == 'multinomial':
            return self.multinomial_loss(bow, word_dist)
        elif self.loss_type == 'poisson':
            return self.poisson_loss(bow, word_dist)
        else:
            raise ValueError(f"Unknown loss type: {self.loss_type}")

print("ReconstructionLoss class defined (multinomial and Poisson options).")


ReconstructionLoss class defined (multinomial and Poisson options).


---
## 7. Complete DETM Model

### 7.1 Full DETM Implementation

In [ ]:
class DETM(nn.Module):
    """
    Complete Dynamic Embedded Topic Model with Full Architecture.
    
    Combines three distinct variational families:
    1. TemporalBaselineEncoder: LSTM structured inference for η_t
    2. DocumentTopicEncoder:    MLP amortized inference for θ_d (logistic-normal)
    3. Mean-field parameters:   Trainable α^(t)  (T × K × L)
    
    ELBO: E_q[log p(w|θ,α,ρ)] - KL(q(θ)||p(θ)) - KL(q(η)||p(η)) - KL(q(α)||p(α))
    
    Priors (Gaussian random walks from Dieng et al. 2019):
        p(η_0) = N(0, δ²I),  p(η_t | η_{t-1}) = N(η_{t-1}, δ²I)   δ² = ETA_PRIOR_VARIANCE
        p(α_0) = N(0, γ²I),  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)   γ² = ALPHA_PRIOR_VARIANCE
        p(θ_d) = Logistic-Normal(η_{t_d}, I)   (η is the PRIOR MEAN, not just conditioning)
    
    Loss scaling — matches Dieng et al. 2019 reference implementation (detm.py):
        coeff = num_train_docs / batch_size
        recon_loss = per_doc_nll.sum() * coeff        # corpus scale
        kl_theta   = per_doc_kl.sum()  * coeff        # corpus scale
        kl_eta     = raw global sum over T×K           # corpus scale (no coeff needed)
        kl_alpha   = raw global sum over T×K×L         # corpus scale (no coeff needed)
        loss       = recon_loss + kl_theta + kl_eta + kl_alpha  # all corpus-scale, no weights
    
    The LSTM is re-run on every forward call (matching Dieng et al. reference impl).
    This is cheap (T~50 steps) vs encoding a batch of documents, and keeps the full
    gradient graph intact — no graph-reuse RuntimeErrors.
    """
    
    def __init__(
        self,
        config,
        word_embeddings: torch.Tensor,
        num_time_steps: int,
        avg_bow_timeline: np.ndarray,
        num_train_docs: int = 1
    ):
        super().__init__()
        
        self.config = config
        self.num_time_steps = num_time_steps
        # D: number of training documents — used to scale coeff = D / batch_size
        self.num_train_docs = num_train_docs
        
        vocab_size, embedding_dim = word_embeddings.shape
        
        # Store average BoW timeline for LSTM (as buffer, not parameter)
        self.register_buffer('avg_bow_timeline', torch.FloatTensor(avg_bow_timeline))
        
        # Component 1: Temporal Baseline Encoder
        self.temporal_baseline_encoder = TemporalBaselineEncoder(
            vocab_size=vocab_size,
            num_topics=config.NUM_TOPICS,
            compression_dim=config.COMPRESSION_DIM,
            lstm_layers=config.LSTM_LAYERS,
            lstm_hidden=config.LSTM_HIDDEN,
            delta_sq=config.ETA_PRIOR_VARIANCE  # δ²
        )
        
        # Component 2: Document Topic Encoder
        self.doc_encoder = DocumentTopicEncoder(
            vocab_size=vocab_size,
            num_topics=config.NUM_TOPICS,
            hidden_dim=config.DOC_HIDDEN_DIM,
            dropout=config.DOC_DROPOUT
        )
        
        # Component 3: Topic Embeddings (mean-field, no neural network)
        # Shape: (T, K, L) — time steps × topics × embedding dimension
        self.alpha_mu = nn.Parameter(
            torch.randn(num_time_steps, config.NUM_TOPICS, embedding_dim) * config.INIT_ALPHA_STD
        )
        self.alpha_logvar = nn.Parameter(
            torch.ones(num_time_steps, config.NUM_TOPICS, embedding_dim) * math.log(config.ALPHA_PRIOR_VARIANCE)
            + torch.randn(num_time_steps, config.NUM_TOPICS, embedding_dim) * 0.1
        )  # Prior-centered: var starts at γ²=ALPHA_PRIOR_VARIANCE; small randn breaks symmetry
        
        # Component 4: Decoder — word embeddings trainable by default (matches original).
        self.decoder = DETMDecoder(
            num_topics=config.NUM_TOPICS,
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            word_embeddings=word_embeddings,
            train_word_embeddings=getattr(config, 'TRAIN_WORD_EMBEDDINGS', True)
        )
        
        self.recon_loss_fn = ReconstructionLoss(loss_type='multinomial')
    
    def sample_alpha(self, time_indices: torch.Tensor) -> torch.Tensor:
        """
        Sample topic embeddings α^(t) via reparameterization.

        At eval time, returns the posterior mean (no noise) for stable inference.

        At train time, samples *once per unique time step* so all documents at
        the same time step share the same α^(t) sample — matching the global-latent
        semantics of α (one set of topic embeddings per year, not per document).
        Per-document independent noise would add destructive gradient variance
        since α^(t) is identical for every document from year t.
        """
        if not self.training:
            return self.alpha_mu[time_indices]  # (B, K, L) — posterior mean, no noise

        # Allocate output buffer (inherits shape, device, dtype from alpha_mu)
        alpha_samples = torch.empty(
            len(time_indices), self.alpha_mu.shape[1], self.alpha_mu.shape[2],
            device=self.alpha_mu.device, dtype=self.alpha_mu.dtype
        )

        # Sample ONCE per unique time step; broadcast to all docs at that time step
        for t in time_indices.unique():
            mask     = time_indices == t
            mu_t     = self.alpha_mu[t]               # (K, L)
            logvar_t = self.alpha_logvar[t]            # (K, L)
            std_t    = torch.exp(0.5 * logvar_t)       # (K, L)
            eps      = torch.randn_like(std_t)          # (K, L) — ONE sample for all docs at t
            alpha_samples[mask] = mu_t + eps * std_t    # broadcast to all docs at time t

        return alpha_samples  # (B, K, L)
    
    def _reparameterize_alpha(self, mu, logvar):
        """Sample from q(α)=N(μ, exp(logvar)·I) at train time; return μ at eval."""
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + torch.randn_like(std) * std
        return mu

    def compute_alpha_kl(self) -> torch.Tensor:
        """
        KL divergence for topic embedding mean-field parameters.

        Uses the Monte Carlo (MC) estimator matching Dieng et al. 2019:
        for t > 0, the prior mean is the *sampled* α_{t-1} (not μ_{t-1}).
        This matches the original `get_alpha()` which uses `alphas[t-1]` as `p_mu`.

        The analytical expectation (adding σ²_{t-1}/γ² per element) creates a
        massive initial KL (σ²/γ² ≈ 200 per element × 750K elements), causing
        over-regularization that collapses topics early in training.

        p(α_0) = N(0, γ²I),  p(α_t | α_{t-1}) = N(α_{t-1}, γ²I)

        Returns a raw sum over all T×K×L elements (corpus scale).
        """
        import math
        gamma_sq = self.config.ALPHA_PRIOR_VARIANCE  # γ²
        log_gamma = math.log(gamma_sq)
        total_kl = 0.0

        # Sample α for all time steps (MC samples used as prior means for t+1)
        alpha_samples = [
            self._reparameterize_alpha(self.alpha_mu[t], self.alpha_logvar[t])
            for t in range(self.num_time_steps)
        ]

        # t = 0: prior is N(0, I)
        var_0 = self.alpha_logvar[0].exp()
        total_kl += 0.5 * torch.sum(
        var_0 + self.alpha_mu[0].pow(2) - 1.0 - self.alpha_logvar[0]
        )

        # t > 0: prior mean = sampled α_{t-1} (MC estimate — matches original)
        for t in range(1, self.num_time_steps):
            var_t = self.alpha_logvar[t].exp()
            alpha_prev = alpha_samples[t - 1]  # sampled prior mean; gradient flows back
            total_kl += 0.5 * torch.sum(
                var_t / gamma_sq
                + (self.alpha_mu[t] - alpha_prev).pow(2) / gamma_sq
                - 1.0
                - (self.alpha_logvar[t] - log_gamma)
            )

        return total_kl
    
    def forward(
        self,
        bow: torch.Tensor,
        time_indices: torch.Tensor,
        compute_loss: bool = True
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass through DETM.
        
        The LSTM is re-run on every forward call so that the full computation
        graph is fresh each batch — this matches the original Dieng et al.
        implementation and avoids the 'backward through graph a second time' error.
        
        Args:
            bow:          Bag-of-words (batch_size, V)
            time_indices: Time step index per document (batch_size,)
            compute_loss: Whether to compute ELBO loss terms
        
        Returns:
            Dict with theta, word_dist, and optionally all loss components.
            All loss terms are on corpus scale (matching Dieng et al. 2019).
        """
        # Run LSTM over all time steps — fresh graph every call, all gradients intact
        eta_timeline, kl_eta = self.temporal_baseline_encoder(self.avg_bow_timeline)
        
        # Look up η_{t_d} for each document (live tensor, gradient flows to LSTM)
        eta_batch = eta_timeline[time_indices]  # (B, K)
        
        # Encode: (BoW, η) → θ (logistic-normal)
        theta, theta_mu, theta_logvar = self.doc_encoder(bow, eta_batch)  # (B, K)
        
        # Sample α^(t) for each document's time step
        alpha_batch = self.sample_alpha(time_indices)  # (B, K, L)
        
        # Decode: (θ, α) → word distribution
        word_dist = self.decoder(theta, alpha_batch)  # (B, V)
        
        output = {
            'theta': theta,
            'theta_mu': theta_mu,
            'theta_logvar': theta_logvar,
            'word_dist': word_dist,
            'alpha': alpha_batch
        }
        
        if compute_loss:
            bsz = bow.shape[0]
            # coeff = D/B scales per-doc sums to corpus magnitude (Dieng et al. 2019)
            coeff = self.num_train_docs / bsz

            # 1. Reconstruction: per-doc NLL → sum → scale to corpus
            #    recon_loss_fn returns (B,) thanks to the no-mean forward()
            recon_loss = self.recon_loss_fn(bow, word_dist).sum() * coeff

            # 2. KL(q(θ) || p(θ=η_t, I)): per-doc KL → sum → scale to corpus
            #    Prior mean is η_batch (not zero) — η sets the document-level prior.
            var_theta = theta_logvar.exp()
            kl_theta = 0.5 * torch.sum(
                (theta_mu - eta_batch).pow(2) + var_theta - theta_logvar - 1.0,
                dim=-1
            ).sum() * coeff

            # 3. KL(q(η) || p(η)) — raw global sum over T×K (already corpus scale)
            # kl_eta comes from temporal_baseline_encoder, no further scaling needed

            # 4. KL(q(α) || p(α)) — raw global sum over T×K×L (already corpus scale)
            kl_alpha = self.compute_alpha_kl()

            # NELBO: all four terms on corpus scale — no loss weights (matches original)
            loss = recon_loss + kl_theta + kl_eta + kl_alpha

            output.update({
                'loss': loss,
                'recon_loss': recon_loss,
                'kl_theta': kl_theta,
                'kl_eta': kl_eta,
                'kl_alpha': kl_alpha
            })
        
        return output
    
    def get_topics(self, time_idx: int = -1, top_n: int = 10) -> List[List[Tuple[str, float]]]:
        """Get top-N words per topic at a given time step."""
        if not hasattr(self, 'idx2word'):
            raise ValueError("idx2word not set. Assign model.idx2word before calling get_topics()")
        
        with torch.no_grad():
            if time_idx == -1:
                time_idx = self.num_time_steps - 1
            alpha = self.alpha_mu[time_idx]      # (K, L) — use mean for inspection
            beta  = self.decoder.get_beta(alpha) # (K, V)
            beta  = beta.cpu().numpy()
        
        topics = []
        for k in range(self.config.NUM_TOPICS):
            top_idx   = beta[k].argsort()[-top_n:][::-1]
            top_words = [(self.idx2word[idx], float(beta[k, idx])) for idx in top_idx]
            topics.append(top_words)
        return topics
    
    def get_document_topics(self, bow: torch.Tensor, time_indices: torch.Tensor) -> np.ndarray:
        """Get topic distributions for a batch of documents."""
        self.eval()
        device = next(self.parameters()).device
        bow = bow.to(device)
        time_indices = time_indices.to(device)
        with torch.no_grad():
            eta_timeline, _ = self.temporal_baseline_encoder(self.avg_bow_timeline)
            eta_batch = eta_timeline[time_indices]
            theta, _, _ = self.doc_encoder(bow, eta_batch)
        return theta.cpu().numpy()


print("DETM class defined.")


DETM class defined.


---
## 8. Dataset and DataLoader

In [ ]:
class DETMDataset(Dataset):
    """
    PyTorch Dataset for DETM with temporal information.

    Accepts a scipy sparse CSR matrix for bow_matrix to avoid materialising
    the full dense (N × V) float32 tensor in RAM. Each row is converted to a
    dense 1-D tensor on demand inside __getitem__ — only batch_size rows are
    ever dense at once, regardless of dataset size.
    """

    def __init__(
        self,
        bow_matrix,          # scipy CSR or numpy ndarray — kept as-is, NOT converted to Tensor
        time_indices: np.ndarray,
        metadata: pd.DataFrame = None
    ):
        self.bow_matrix = bow_matrix          # sparse CSR: ~100x less RAM than dense
        self.time_indices = torch.LongTensor(time_indices)
        self.metadata = metadata
        self._is_sparse = sparse.issparse(bow_matrix)

    def __len__(self) -> int:
        return self.bow_matrix.shape[0]

    def __getitem__(self, idx: int) -> Dict:
        if self._is_sparse:
            # Convert single sparse row to dense 1-D tensor on the fly.
            # .toarray() returns shape (1, V); .ravel() makes it (V,).
            bow = torch.FloatTensor(
                np.asarray(self.bow_matrix[idx].toarray()).ravel()
            )
        else:
            bow = torch.FloatTensor(self.bow_matrix[idx])

        item = {'bow': bow, 'time_idx': self.time_indices[idx]}

        if self.metadata is not None:
            item['metadata'] = self.metadata.iloc[idx].to_dict()

        return item

def create_dataloaders(
    processed_data: Dict,
    batch_size: int,
    train_split: float = 0.85,
    val_split: float = 0.05
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train/val/test dataloaders with temporal information.
    
    Args:
        processed_data: Output from DataPreprocessor (includes temporal_info)
        batch_size: Batch size
        train_split: Fraction for training
        val_split: Fraction for validation
    
    Returns:
        train_loader, val_loader, test_loader
    """
    bow_matrix = processed_data['bow_matrix']
    # Drop the heavy 'text' column — sentences are ~100 chars each but 450K rows
    # still adds up; year/country are all that's needed for temporal analysis.
    _meta = processed_data['metadata']
    _keep_cols = [c for c in _meta.columns if c != 'text']
    metadata = _meta[_keep_cols]
    time_indices = processed_data['temporal_info']['doc_to_time']
    
    # Shuffle before splitting so all time periods appear in every split.
    # Data is sorted by year; a sequential cut puts recent years exclusively in val/test.
    n = bow_matrix.shape[0]
    rng = np.random.default_rng(42)
    perm = rng.permutation(n)
    bow_matrix   = bow_matrix[perm]
    time_indices = time_indices[perm]
    if metadata is not None:
        metadata = metadata.iloc[perm].reset_index(drop=True)

    n_train = int(n * train_split)
    n_val = int(n * val_split)
    
    # Create datasets
    train_dataset = DETMDataset(
        bow_matrix[:n_train],
        time_indices[:n_train],
        metadata.iloc[:n_train] if metadata is not None else None
    )
    
    val_dataset = DETMDataset(
        bow_matrix[n_train:n_train+n_val],
        time_indices[n_train:n_train+n_val],
        metadata.iloc[n_train:n_train+n_val] if metadata is not None else None
    )
    
    test_dataset = DETMDataset(
        bow_matrix[n_train+n_val:],
        time_indices[n_train+n_val:],
        metadata.iloc[n_train+n_val:] if metadata is not None else None
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    print(f"\nDataset splits:")
    print(f"  Train: {len(train_dataset)} documents")
    print(f"  Val: {len(val_dataset)} documents")
    print(f"  Test: {len(test_dataset)} documents")
    
    return train_loader, val_loader, test_loader

print("Dataset classes defined.")

Dataset classes defined.


---
## 9. Training Infrastructure

### 9.1 DETM Trainer Class

In [18]:
class DETMTrainer:
    """
    Training loop for DETM with monitoring and checkpointing.
    Includes TensorBoard integration for real-time visualization.

    Optimizer: Adam with constant lr and wd=1.2e-6 (matches Dieng et al. 2019).
    No LR scheduler — constant learning rate avoids premature convergence.
    Each training run gets a timestamped TensorBoard subdirectory so runs can
    be compared side-by-side in TensorBoard.

    Resume training:
        Pass resume_checkpoint_path to __init__ to restore model weights,
        optimizer state, epoch counter, best_val_loss and history from a
        saved checkpoint — allowing training to continue seamlessly.
    """

    def __init__(
        self,
        model: DETM,
        config: Config,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        log_dir: Optional[str] = None,
        resume_checkpoint_path: Optional[str] = None,
        resume_lr: Optional[float] = None,
    ):
        from datetime import datetime

        self.model = model.to(device)
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        if log_dir is None:
            log_dir = str(config.OUTPUTS_DIR / 'tensorboard_logs' / timestamp)
        self.writer = SummaryWriter(log_dir=log_dir)
        print(f"TensorBoard run: {timestamp}")
        print(f"  Logging to: {log_dir}")
        print(f"  View with: tensorboard --logdir={config.OUTPUTS_DIR / 'tensorboard_logs'}")

        lr = resume_lr if resume_lr is not None else config.LEARNING_RATE
        self.optimizer = Adam(
            model.parameters(),
            lr=lr,
            weight_decay=config.WEIGHT_DECAY
        )

        # Tracking
        self.global_step = 0
        self.start_epoch = 1
        self.history = {
            'train_loss': [], 'train_recon': [], 'train_kl_theta': [],
            'train_kl_eta': [], 'train_kl_alpha': [],
            'val_loss': [], 'val_recon': [], 'val_kl_theta': [],
            'val_kl_eta': [], 'val_kl_alpha': [],
            'learning_rate': []
        }
        self.best_val_loss = float('inf')
        self.patience_counter = 0

        # ── Resume from checkpoint ────────────────────────────────────────────
        if resume_checkpoint_path is not None:
            ckpt_path = Path(resume_checkpoint_path)
            if not ckpt_path.exists():
                raise FileNotFoundError(f"Resume checkpoint not found: {ckpt_path}")
            ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
            self.model.load_state_dict(ckpt['model_state_dict'])
            self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            # Override LR in optimizer if resume_lr was specified
            if resume_lr is not None:
                for pg in self.optimizer.param_groups:
                    pg['lr'] = resume_lr
            self.best_val_loss = ckpt.get('best_val_loss', float('inf'))
            self.start_epoch = ckpt.get('epoch', 0) + 1
            saved_history = ckpt.get('history', {})
            for k in self.history:
                self.history[k] = saved_history.get(k, [])
            self.global_step = sum(
                len(self.train_loader) for _ in range(self.start_epoch - 1)
            )
            print(f"\n✓ Resuming from checkpoint: {ckpt_path}")
            print(f"  Last completed epoch : {self.start_epoch - 1}")
            print(f"  Continuing from epoch: {self.start_epoch}")
            print(f"  Best val loss so far : {self.best_val_loss:.4f}")
            print(f"  Learning rate        : {lr:.6f}")
        # ─────────────────────────────────────────────────────────────────────

    def train_epoch(self) -> Dict[str, float]:
        """Train for one epoch. NaN batches are skipped to prevent model corruption."""
        self.model.train()

        total_loss = 0
        total_recon = 0
        total_kl_theta = 0
        total_kl_eta = 0
        total_kl_alpha = 0
        num_batches = 0
        nan_batches = 0

        pbar = tqdm(self.train_loader, desc="Training")
        for batch in pbar:
            bow = batch['bow'].to(self.device)
            time_indices = batch['time_idx'].to(self.device)

            output = self.model(bow, time_indices, compute_loss=True)

            loss = output['loss']

            # ── NaN guard ────────────────────────────────────────────────────
            if not torch.isfinite(loss):
                nan_batches += 1
                self.optimizer.zero_grad()  # clear any partial grads
                if nan_batches <= 3:
                    print(f"\n  ⚠ NaN/Inf loss in batch — skipping (total skipped: {nan_batches})")
                self.global_step += 1
                continue
            # ─────────────────────────────────────────────────────────────────

            recon_loss = output['recon_loss']
            kl_theta   = output['kl_theta']
            kl_eta     = output['kl_eta']
            kl_alpha   = output['kl_alpha']

            self.optimizer.zero_grad()
            loss.backward()

            # Gradient clipping (disabled if CLIP_GRAD=0, per original default)
            if self.config.CLIP_GRAD > 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config.CLIP_GRAD
                )
            else:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), float('inf')
                )
            if not torch.isfinite(grad_norm):
                nan_batches += 1
                self.optimizer.zero_grad()
                if nan_batches <= 3:
                    print(f"\n  ⚠ NaN gradient norm — skipping update (total: {nan_batches})")
                self.global_step += 1
                continue

            self.optimizer.step()

            total_loss     += loss.item()
            total_recon    += recon_loss.item()
            total_kl_theta += kl_theta.item()
            total_kl_eta   += kl_eta.item()
            total_kl_alpha += kl_alpha.item()
            num_batches += 1

            if self.global_step % 10 == 0:
                self.writer.add_scalar('Batch/Loss', loss.item(), self.global_step)
                self.writer.add_scalar('Batch/Reconstruction', recon_loss.item(), self.global_step)
                self.writer.add_scalar('Batch/KL_theta', kl_theta.item(), self.global_step)
                self.writer.add_scalar('Batch/KL_eta', kl_eta.item(), self.global_step)
                self.writer.add_scalar('Batch/KL_alpha', kl_alpha.item(), self.global_step)
                self.writer.add_scalar('Batch/Grad_Norm', grad_norm.item(), self.global_step)

            self.global_step += 1

            pbar.set_postfix({
                'loss':  f"{loss.item():.2f}",
                'recon': f"{recon_loss.item():.2f}",
                'KL_θ':  f"{kl_theta.item():.2f}",
                'KL_η':  f"{kl_eta.item():.2f}",
                'KL_α':  f"{kl_alpha.item():.2f}",
                'skip':  nan_batches,
            })

        if nan_batches > 0:
            print(f"  ⚠ Epoch had {nan_batches} skipped NaN/Inf batches out of "
                  f"{num_batches + nan_batches} total")

        if num_batches == 0:
            # Entire epoch was NaN — return NaN so caller can detect and stop
            return {k: float('nan') for k in [
                'train_loss', 'train_recon', 'train_kl_theta', 'train_kl_eta', 'train_kl_alpha'
            ]}

        return {
            'train_loss':      total_loss / num_batches,
            'train_recon':     total_recon / num_batches,
            'train_kl_theta':  total_kl_theta / num_batches,
            'train_kl_eta':    total_kl_eta / num_batches,
            'train_kl_alpha':  total_kl_alpha / num_batches,
        }

    @torch.no_grad()
    def validate(self) -> Dict[str, float]:
        """Validate on validation set."""
        self.model.eval()

        total_loss = 0
        total_recon = 0
        total_kl_theta = 0
        total_kl_eta = 0
        total_kl_alpha = 0
        num_batches = 0

        for batch in tqdm(self.val_loader, desc="Validation"):
            bow = batch['bow'].to(self.device)
            time_indices = batch['time_idx'].to(self.device)

            output = self.model(bow, time_indices, compute_loss=True)
            if not torch.isfinite(output['loss']):
                continue

            total_loss     += output['loss'].item()
            total_recon    += output['recon_loss'].item()
            total_kl_theta += output['kl_theta'].item()
            total_kl_eta   += output['kl_eta'].item()
            total_kl_alpha += output['kl_alpha'].item()
            num_batches += 1

        if num_batches == 0:
            return {k: float('nan') for k in [
                'val_loss', 'val_recon', 'val_kl_theta', 'val_kl_eta', 'val_kl_alpha'
            ]}

        return {
            'val_loss':     total_loss / num_batches,
            'val_recon':    total_recon / num_batches,
            'val_kl_theta': total_kl_theta / num_batches,
            'val_kl_eta':   total_kl_eta / num_batches,
            'val_kl_alpha': total_kl_alpha / num_batches,
        }

    def save_checkpoint(self, epoch: int, is_best: bool = False):
        """Save model checkpoint."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_loss': self.best_val_loss,
            'history': self.history,
            'config': self.config
        }

        if epoch % self.config.SAVE_EVERY == 0:
            path = self.config.MODELS_DIR / f'detm_epoch_{epoch}.pt'
            torch.save(checkpoint, path)
            print(f"Checkpoint saved: {path}")

        if is_best:
            path = self.config.MODELS_DIR / 'detm_best.pt'
            torch.save(checkpoint, path)
            print(f"Best model saved: {path}")

    def train(self, num_epochs: int = None):
        """Complete training loop. Starts from self.start_epoch (supports resume)."""
        num_epochs = num_epochs or self.config.NUM_EPOCHS
        current_lr = self.optimizer.param_groups[0]['lr']

        print("\n" + "=" * 60)
        print("TRAINING DETM")
        print("=" * 60)
        print(f"Starting epoch  : {self.start_epoch}")
        print(f"Total epochs    : {num_epochs}")
        print(f"Learning rate   : {current_lr}")
        print(f"Optimizer       : Adam  |  weight_decay: {self.config.WEIGHT_DECAY}")
        print(f"Grad clip       : {self.config.CLIP_GRAD} (0=disabled)")
        print(f"LR annealing    : ÷{self.config.ANNEAL_FACTOR:.0f} every {self.config.ANNEAL_PATIENCE} non-improvement epochs | min: {self.config.MIN_LR:.0e}")
        print(f"Device          : {self.device}")
        print("=" * 60 + "\n")

        for epoch in range(self.start_epoch, num_epochs + 1):
            current_lr = self.optimizer.param_groups[0]['lr']  # refresh (handles annealing)
            print(f"\nEpoch {epoch}/{num_epochs}")
            print("-" * 60)

            train_metrics = self.train_epoch()

            # If the entire epoch was NaN, abort — model state is unrecoverable
            if not all(np.isfinite(v) for v in train_metrics.values()):
                print(f"\n✗ Entire epoch {epoch} produced NaN — aborting training.")
                print("  Reload the best checkpoint and retry with a lower learning rate.")
                break

            val_metrics = self.validate()

            self.history['train_loss'].append(train_metrics['train_loss'])
            self.history['train_recon'].append(train_metrics['train_recon'])
            self.history['train_kl_theta'].append(train_metrics['train_kl_theta'])
            self.history['train_kl_eta'].append(train_metrics['train_kl_eta'])
            self.history['train_kl_alpha'].append(train_metrics['train_kl_alpha'])
            self.history['val_loss'].append(val_metrics['val_loss'])
            self.history['val_recon'].append(val_metrics['val_recon'])
            self.history['val_kl_theta'].append(val_metrics['val_kl_theta'])
            self.history['val_kl_eta'].append(val_metrics['val_kl_eta'])
            self.history['val_kl_alpha'].append(val_metrics['val_kl_alpha'])
            self.history['learning_rate'].append(current_lr)

            self.writer.add_scalar('Epoch/Train/Loss', train_metrics['train_loss'], epoch)
            self.writer.add_scalar('Epoch/Train/Reconstruction', train_metrics['train_recon'], epoch)
            self.writer.add_scalar('Epoch/Train/KL_theta', train_metrics['train_kl_theta'], epoch)
            self.writer.add_scalar('Epoch/Train/KL_eta', train_metrics['train_kl_eta'], epoch)
            self.writer.add_scalar('Epoch/Train/KL_alpha', train_metrics['train_kl_alpha'], epoch)
            self.writer.add_scalar('Epoch/Val/Loss', val_metrics['val_loss'], epoch)
            self.writer.add_scalar('Epoch/Val/Reconstruction', val_metrics['val_recon'], epoch)
            self.writer.add_scalar('Epoch/Val/KL_theta', val_metrics['val_kl_theta'], epoch)
            self.writer.add_scalar('Epoch/Val/KL_eta', val_metrics['val_kl_eta'], epoch)
            self.writer.add_scalar('Epoch/Val/KL_alpha', val_metrics['val_kl_alpha'], epoch)
            self.writer.add_scalar('Epoch/Learning_Rate', current_lr, epoch)

            total_norm = 0.0
            for p in self.model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            self.writer.add_scalar('Epoch/Gradient_Norm', total_norm ** 0.5, epoch)

            # Alpha posterior diagnostics — track collapse (PostVar→0) or explosion (Logvar→+∞)
            with torch.no_grad():
                al = self.model.alpha_logvar.detach().cpu()
                am = self.model.alpha_mu.detach().cpu()
                post_var = al.exp()
                self.writer.add_scalar('Alpha/PostVar_mean',  post_var.mean().item(), epoch)
                self.writer.add_scalar('Alpha/PostVar_std',   post_var.std().item(),  epoch)
                self.writer.add_scalar('Alpha/Logvar_mean',   al.mean().item(),       epoch)
                self.writer.add_scalar('Alpha/Logvar_min',    al.min().item(),        epoch)
                self.writer.add_scalar('Alpha/Logvar_max',    al.max().item(),        epoch)
                self.writer.add_scalar('Alpha/Mu_mean',       am.mean().item(),       epoch)
                self.writer.add_scalar('Alpha/Mu_norm_mean',  am.norm(dim=-1).mean().item(), epoch)

            print(f"\nEpoch {epoch} Summary:")
            print(f"  Train Loss: {train_metrics['train_loss']:.4f} "
                  f"(Recon: {train_metrics['train_recon']:.4f}, "
                  f"KL_θ: {train_metrics['train_kl_theta']:.4f}, "
                  f"KL_η: {train_metrics['train_kl_eta']:.4f}, "
                  f"KL_α: {train_metrics['train_kl_alpha']:.4f})")
            print(f"  Val Loss: {val_metrics['val_loss']:.4f} "
                  f"(Recon: {val_metrics['val_recon']:.4f}, "
                  f"KL_θ: {val_metrics['val_kl_theta']:.4f}, "
                  f"KL_η: {val_metrics['val_kl_eta']:.4f}, "
                  f"KL_α: {val_metrics['val_kl_alpha']:.4f})")
            print(f"  Learning Rate: {current_lr:.6f}")

            is_best = val_metrics['val_loss'] < self.best_val_loss
            if is_best:
                self.best_val_loss = val_metrics['val_loss']
                self.patience_counter = 0
                print(f"  ✓ New best validation loss: {self.best_val_loss:.4f}")
            else:
                self.patience_counter += 1
                # LR annealing: divide by ANNEAL_FACTOR every ANNEAL_PATIENCE non-improvement epochs
                if self.patience_counter % self.config.ANNEAL_PATIENCE == 0:
                    for pg in self.optimizer.param_groups:
                        pg['lr'] /= self.config.ANNEAL_FACTOR
                    current_lr = self.optimizer.param_groups[0]['lr']
                    print(f"  No improvement for {self.patience_counter} epochs "
                          f"→ LR annealed to {current_lr:.2e}")
                else:
                    print(f"  No improvement for {self.patience_counter} epochs")

            self.save_checkpoint(epoch, is_best=is_best)

            # Stop if LR has dropped below minimum threshold
            current_lr = self.optimizer.param_groups[0]['lr']
            if current_lr < self.config.MIN_LR:
                print(f"\n  LR {current_lr:.2e} is below minimum {self.config.MIN_LR:.0e}. Stopping.")
                break

            if self.patience_counter >= self.config.PATIENCE:
                print(f"\nEarly stopping triggered after {epoch} epochs")
                break

        self.writer.close()
        print(f"\nTensorBoard logs saved. View with: tensorboard --logdir={self.writer.log_dir}")
        print("\n" + "=" * 60)
        print("TRAINING COMPLETE")
        print("=" * 60)
        print(f"Best validation loss: {self.best_val_loss:.4f}")
        print("=" * 60)

        return self.history


print("DETMTrainer class defined.")


DETMTrainer class defined.


---
## 10. Evaluation Metrics

### 10.1 Topic Coherence and Quality Metrics

In [ ]:
class TopicEvaluator:
    """
    Evaluates topic model quality using various metrics.
    
    Metrics:
    - Topic coherence (C_v, C_npmi)
    - Topic diversity
    - Perplexity
    """
    
    def __init__(self, tokens_list: List[List[str]], vocabulary: List[str]):
        self.tokens_list = tokens_list
        self.vocabulary = vocabulary
        
        # Create Gensim dictionary and corpus
        self.dictionary = Dictionary(tokens_list)
        self.corpus = [self.dictionary.doc2bow(doc) for doc in tokens_list]
    
    def compute_coherence(
        self,
        topics: List[List[str]],
        coherence_type: str = 'c_v'
    ) -> float:
        """
        Compute topic coherence score.
        
        Args:
            topics: List of topics, each is a list of words
            coherence_type: 'c_v' or 'c_npmi'
        
        Returns:
            Average coherence score
        """
        cm = CoherenceModel(
            topics=topics,
            texts=self.tokens_list,
            dictionary=self.dictionary,
            coherence=coherence_type
        )
        
        return cm.get_coherence()
    
    def compute_topic_diversity(self, topics: List[List[str]]) -> float:
        """
        Compute topic diversity.
        
        Measures what fraction of unique words appear in the top-N words
        across all topics. Higher is better (less redundancy).
        
        Args:
            topics: List of topics, each is a list of words
        
        Returns:
            Diversity score (0 to 1)
        """
        unique_words = set()
        total_words = 0
        
        for topic in topics:
            unique_words.update(topic)
            total_words += len(topic)
        
        diversity = len(unique_words) / total_words if total_words > 0 else 0
        return diversity
    
    def evaluate_topics(
        self,
        model: DETM,
        top_n_words: List[int] = [10, 15, 20]
    ) -> Dict[str, float]:
        """
        Comprehensive topic evaluation averaged across multiple time steps.

        Coherence is averaged over 5 evenly-spaced time steps to capture temporal
        evolution. Diversity is reported at the last time step (standard convention).
        """
        results = {}
        T = model.num_time_steps
        # 5 evenly-spaced time steps covering the full temporal range
        time_samples = sorted(set([0, T // 4, T // 2, 3 * T // 4, T - 1]))

        for n in top_n_words:
            cv_scores, npmi_scores = [], []
            for t in time_samples:
                topics_with_probs = model.get_topics(time_idx=t, top_n=n)
                topics = [[word for word, _ in topic] for topic in topics_with_probs]
                cv_scores.append(self.compute_coherence(topics, coherence_type='c_v'))
                npmi_scores.append(self.compute_coherence(topics, coherence_type='c_npmi'))

            results[f'coherence_cv_top{n}'] = float(np.mean(cv_scores))
            results[f'coherence_npmi_top{n}'] = float(np.mean(npmi_scores))

            # Diversity at last time step (conventional reporting)
            topics_last = [[w for w, _ in tp] for tp in model.get_topics(time_idx=-1, top_n=n)]
            results[f'diversity_top{n}'] = self.compute_topic_diversity(topics_last)

        return results
    
    @staticmethod
    def compute_perplexity(
        model: DETM,
        data_loader: DataLoader,
        device: torch.device
    ) -> float:
        """
        Compute perplexity matching Dieng et al. 2019 (get_completion_ppl).

        Computes the arithmetic mean of per-word NLL across all batches,
        then takes exp(). This weights each document equally regardless of
        length — directly comparable with numbers reported in the paper.

        Original formula:
            nll_per_doc  = -(bow * log(word_dist)).sum(-1)  # (B,)
            per_word_nll = nll_per_doc / doc_length          # (B,)
            loss         = per_word_nll.mean()               # scalar mean over batch
            perplexity   = exp(mean over all batches of loss)
        """
        import math as _math
        model.eval()
        acc_loss = 0.0
        cnt = 0
        with torch.no_grad():
            for batch in data_loader:
                bow = batch['bow'].to(device)
                time_indices = batch['time_idx'].to(device)
                output = model(bow, time_indices, compute_loss=False)
                word_dist = torch.clamp(output['word_dist'], min=1e-10)
                nll = -(bow * torch.log(word_dist)).sum(dim=-1)   # (B,) per-doc NLL
                sums = bow.sum(dim=-1)                             # (B,) doc lengths
                per_word_nll = nll / (sums + 1e-10)               # (B,) per-word NLL
                acc_loss += per_word_nll.mean().item()             # mean over batch
                cnt += 1
        return float(_math.exp(acc_loss / cnt)) if cnt > 0 else float('inf')

print("TopicEvaluator class defined.")


TopicEvaluator class defined.


---
## 12. Main Execution Pipeline

### Run complete DETM workflow

In [ ]:
# ── Pipeline control ──────────────────────────────────────────────────────────
# LOAD_PRETRAINED = True   → skip training, load best checkpoint for evaluation
# RESUME_TRAINING = True   → continue training from CHECKPOINT_PATH
#                            (restores weights, optimizer state, epoch, history)
# Both False               → train from scratch
LOAD_PRETRAINED  = False
RESUME_TRAINING  = False                     # ← set True to resume from epoch 52
CHECKPOINT_PATH  = config.MODELS_DIR / 'detm_best.pt'
RESUME_LR        = 0.001                       # lower LR when resuming to avoid NaN recurrence
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Load and preprocess data
df_raw = load_and_explore_data()
preprocessor = DataPreprocessor(config)
processed_data = preprocessor.preprocess_corpus(df_raw)

# Step 2: Generate embeddings (cached after first run)
embedding_gen = EmbeddingGenerator(config)
word_embeddings = embedding_gen.generate_vocabulary_embeddings(
    preprocessor.vocabulary,
    tokens_list=processed_data['tokens_list'],
)
word_embeddings_tensor = torch.FloatTensor(word_embeddings)

# Step 3: Create dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    processed_data,
    batch_size=config.BATCH_SIZE
)

# Step 4: Initialize model architecture (always required)
model = DETM(
    config,
    word_embeddings_tensor,
    processed_data['temporal_info']['num_time_steps'],
    processed_data['temporal_info']['avg_bow_per_time'],
    num_train_docs=len(train_loader.dataset)
)
model.idx2word = preprocessor.idx2word
print(f"\nModel initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"  num_train_docs = {model.num_train_docs}")

# Step 5: Train / resume / load
if LOAD_PRETRAINED:
    # ── Evaluation only ───────────────────────────────────────────────────────
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"No checkpoint found at {CHECKPOINT_PATH}. "
            "Train the model first (set LOAD_PRETRAINED = False)."
        )
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    history = checkpoint.get('history', {})
    print(f"\nLoaded checkpoint: {CHECKPOINT_PATH}")
    print(f"  Saved at epoch  : {checkpoint.get('epoch', 'unknown')}")
    print(f"  Best val loss   : {checkpoint.get('best_val_loss', float('nan')):.4f}")

elif RESUME_TRAINING:
    # ── Resume training from checkpoint ───────────────────────────────────────
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"No checkpoint to resume from: {CHECKPOINT_PATH}"
        )
    model = model.to(device)
    trainer = DETMTrainer(
        model, config, train_loader, val_loader, device,
        resume_checkpoint_path=str(CHECKPOINT_PATH),
        resume_lr=RESUME_LR,
    )
    history = trainer.train()

    # Reload best checkpoint weights for evaluation
    if CHECKPOINT_PATH.exists():
        best_ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        model.load_state_dict(best_ckpt['model_state_dict'])
        print(f"\n✓ Reloaded best checkpoint for evaluation"
              f" (epoch {best_ckpt.get('epoch', '?')},"
              f" val_loss={best_ckpt.get('best_val_loss', float('nan')):.4f})")

else:
    # ── Train from scratch ────────────────────────────────────────────────────
    model = model.to(device)
    trainer = DETMTrainer(model, config, train_loader, val_loader, device)
    history = trainer.train()

    if CHECKPOINT_PATH.exists():
        best_ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        model.load_state_dict(best_ckpt['model_state_dict'])
        print(f"\n✓ Reloaded best checkpoint for evaluation"
              f" (epoch {best_ckpt.get('epoch', '?')},"
              f" val_loss={best_ckpt.get('best_val_loss', float('nan')):.4f})")
    else:
        print("\n⚠ No best checkpoint found on disk — evaluating last-epoch weights.")

# Step 6: Evaluate
evaluator = TopicEvaluator(processed_data['tokens_list'], preprocessor.vocabulary)
metrics = evaluator.evaluate_topics(model, top_n_words=config.TOP_N_WORDS)
perplexity = evaluator.compute_perplexity(model, test_loader, device)

print("\nEvaluation Metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")
print(f"  Test Perplexity: {perplexity:.4f}")

# Step 7: Save results
results = {
    'config': config.__dict__,
    'metrics': metrics,
    'perplexity': perplexity,
    'history': history
}
with open(config.OUTPUTS_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("\nPipeline complete!")


UN GENERAL DEBATES DATASET

Dataset shape: (7507, 4)

Columns: ['session', 'year', 'country', 'text']

Data types:
session     int64
year        int64
country    object
text       object
dtype: object

Missing values:
session    0
year       0
country    0
text       0
dtype: int64

Year range: 1970 - 2015
Years in dataset: [np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), n

Splitting paragraphs:   0%|          | 0/7507 [00:00<?, ?it/s]

  Speeches → paragraph-docs: 7,507 → 1,377,801
  Speeches with no \n kept as 1 doc (no newlines at all): 0 (0.0%)

Step 2 — Tokenizing and cleaning paragraphs...


Tokenizing:   0%|          | 0/1377801 [00:00<?, ?it/s]

  Documents after length filter (≥5 tokens): 592,355 (removed 785,446)
  Building vocabulary from 473,884 training docs (80% of 592,355 total)
Building vocabulary...


...


Epoch 15 Summary:
  Train Loss: 149.6868 (Recon: 104.0685, KL_θ: 0.3636, KL_η: 0.0227, KL_α: 45.2320)
  Val Loss: 145.1428 (Recon: 103.5512, KL_θ: 0.3541, KL_η: 0.0229, KL_α: 41.2147)
  Learning Rate: 0.000100
  ✓ New best validation loss: 145.1428
Best model saved: ../models/detm_best.pt

Epoch 16/1000
------------------------------------------------------------


Training:   0%|          | 0/677 [00:00<?, ?it/s]

Validation:   0%|          | 0/85 [00:00<?, ?it/s]


Epoch 16 Summary:
  Train Loss: 143.7104 (Recon: 103.9161, KL_θ: 0.3626, KL_η: 0.0228, KL_α: 39.4089)
  Val Loss: 139.6133 (Recon: 103.4624, KL_θ: 0.3684, KL_η: 0.0226, KL_α: 35.7598)
  Learning Rate: 0.000100
  ✓ New best validation loss: 139.6133
Best model saved: ../models/detm_best.pt

Epoch 17/1000
------------------------------------------------------------


Training:   0%|          | 0/677 [00:00<?, ?it/s]

---
## 15. Temporal Analysis — Topic Evolution Over Time

DETM learns a **separate topic-word distribution** β<sub>k,t</sub> for every time step *t*,
capturing how the vocabulary of each topic shifts across the years. This section visualises that
evolution in three complementary ways:

| Cell | What it shows |
|------|---------------|
| **15.1 – Extraction** | Compute the full `(T, K, V)` beta matrix in one shot |
| **15.2 – Tabular view** | Top-10 words per topic at decade years (quick scan) |
| **15.3 – Heatmaps** | Full timeline of word probabilities per topic (orange = high) |
| **15.4 – Trajectories** | Probability curves for defining words across all years |


In [ ]:

# ── 15.1  Extract temporal beta matrix ───────────────────────────────────────
# beta[t, k, v] = P(word v | topic k, time step t)
import matplotlib.pyplot as plt  # ensure matplotlib is imported in this scope

model.eval()
time_steps = processed_data['temporal_info']['time_steps']   # list of year labels
T          = model.num_time_steps
K          = model.config.NUM_TOPICS
vocab_list = [model.idx2word[i] for i in range(len(model.idx2word))]  # ordered vocab

with torch.no_grad():
    beta = model.decoder.get_beta(model.alpha_mu).cpu().numpy()  # (T, K, V)

print(f"Beta shape : {beta.shape}  →  {T} time steps × {K} topics × {len(vocab_list)} words")
print(f"Time range : {time_steps[0]} – {time_steps[-1]}")
print(f"Topics     : {K}")


In [ ]:

# ── 15.2  Tabular view: top-10 words at decade years ─────────────────────────
TOP_N_TABLE = 10

decade_years = [y for y in time_steps if y % 10 == 0]
decade_idxs  = [time_steps.index(y) for y in decade_years]

print(f"Decade years available: {decade_years}\n")
print("=" * 70)

for k in range(K):
    rows = {}
    for yr, t in zip(decade_years, decade_idxs):
        top_idx       = beta[t, k].argsort()[-TOP_N_TABLE:][::-1]
        rows[str(yr)] = [vocab_list[i] for i in top_idx]

    df = pd.DataFrame(rows, index=[f"#{i + 1}" for i in range(TOP_N_TABLE)])
    print(f"\n── Topic {k + 1} ──")
    display(df)


In [ ]:

# ── 15.3  Heatmaps: word probability over the full timeline per topic ─────────
TOP_N_HEAT = 10

ncols = 2
nrows = (K + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes = axes.flatten()

# Downsample time axis so x-tick labels don't overlap
step     = max(1, T // 20)
yr_ticks = time_steps[::step]
yr_idxs  = list(range(0, T, step))

for k in range(K):
    ax = axes[k]
    # Select top-10 words by their MAX probability across all time steps
    # (captures words that "spike" strongly at any point in the timeline)
    max_probs = beta[:, k, :].max(axis=0)                           # (V,)
    top_idx   = max_probs.argsort()[-TOP_N_HEAT:][::-1]
    words     = [vocab_list[i] for i in top_idx]

    heat = beta[np.ix_(yr_idxs, [k], top_idx)][:, 0, :].T          # (TOP_N, len(yr_idxs))
    im   = ax.imshow(heat, aspect='auto', cmap='YlOrRd', interpolation='nearest')

    ax.set_yticks(range(TOP_N_HEAT))
    ax.set_yticklabels(words, fontsize=9)
    ax.set_xticks(range(len(yr_ticks)))
    ax.set_xticklabels(yr_ticks, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Topic {k + 1}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)

for k in range(K, len(axes)):
    axes[k].set_visible(False)

plt.suptitle(
    'Topic–Word Probability Heatmaps over Time\n'
    '(top-10 words per topic by peak probability)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
out_path = config.OUTPUTS_DIR / 'topic_evolution_heatmaps.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")


In [ ]:

# ── 15.4  Trajectory Line Plots: defining word probability over time ──────────
TOP_N_LINES = 5   # words ranked by mean probability across all years

ncols = 2
nrows = (K + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes = axes.flatten()

for k in range(K):
    ax = axes[k]
    mean_probs = beta[:, k, :].mean(axis=0)          # (V,) mean across all T
    top_idx = mean_probs.argsort()[-TOP_N_LINES:][::-1]

    for i in top_idx:
        word   = vocab_list[i]
        probs  = beta[:, k, i]
        ax.plot(time_steps, probs, label=word, linewidth=1.8)

    ax.set_title(f'Topic {k + 1}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('P(word | topic, year)', fontsize=9)
    ax.legend(fontsize=8, loc='upper left', ncol=1)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

for k in range(K, len(axes)):
    axes[k].set_visible(False)

plt.suptitle(
    'Top-5 Word Probability Trajectories per Topic over Time\n'
    '(words ranked by mean probability across all years)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
out_path = config.OUTPUTS_DIR / 'topic_word_trajectories.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")


---
## 16. Preprocessing Comparison — Ours vs. Original DETM

The temporal analysis reveals **generic top words** (e.g. "men") and **no clear climate change topic**.
Both issues likely stem from preprocessing differences vs. the original implementation
([adjidieng/DETM](https://github.com/adjidieng/DETM)).

This section diagnoses the root causes by:
1. Checking which climate-related words survived our preprocessing
2. Comparing our preprocessing pipeline side-by-side with the original
3. Listing concrete recommendations

In [ ]:
# ── 16.1  Vocabulary diagnostics: climate & generic word investigation ────────
climate_probes = [
    'climate', 'warming', 'environment', 'environmental', 'emission',
    'carbon', 'temperature', 'greenhouse', 'sustainable', 'sustainability',
    'renewable', 'fossil', 'deforestation', 'biodiversity', 'ecology',
    'pollution', 'ozone', 'drought', 'flood', 'disaster',
]
generic_probes = ['men', 'man', 'people', 'country', 'time', 'great', 'order']

print("=" * 70)
print("CLIMATE-RELATED WORDS — presence in vocabulary")
print("=" * 70)
vocab_set = set(vocab_list)
for w in climate_probes:
    in_vocab = w in vocab_set
    marker = "✓" if in_vocab else "✗ MISSING"
    extra = ""
    if in_vocab:
        idx = vocab_list.index(w)
        # max probability across all topics and times
        max_p = beta[:, :, idx].max()
        best_k = beta[:, :, idx].max(axis=0).argmax()
        extra = f"  (max P={max_p:.4f}, strongest in Topic {best_k+1})"
    print(f"  {marker:12s}  {w}{extra}")

print(f"\n{'=' * 70}")
print("GENERIC WORDS — probability check")
print("=" * 70)
for w in generic_probes:
    if w in vocab_set:
        idx = vocab_list.index(w)
        max_p = beta[:, :, idx].max()
        mean_p = beta[:, :, idx].mean()
        best_k = beta[:, :, idx].max(axis=0).argmax()
        print(f"  {w:15s}  max P={max_p:.4f}  mean P={mean_p:.6f}  strongest Topic {best_k+1}")
    else:
        print(f"  {w:15s}  not in vocab")

# These words were previously removed by UN_STOPWORDS but are now restored
print(f"\n{'=' * 70}")
print("PREVIOUSLY OVER-FILTERED WORDS — now restored in vocabulary")
print("=" * 70)
previously_removed = [
    'international', 'world', 'countries', 'states', 'government',
    'global', 'nation', 'peoples', 'republic', 'member',
]
for w in previously_removed:
    in_v = w in vocab_set
    status = "✓ in vocab (restored)" if in_v else "✗ missing (below df gate)"
    print(f"  {w:20s}  {status}")


In [ ]:
# ── 16.2  Preprocessing comparison: Ours vs Original DETM ────────────────────
comparison = pd.DataFrame({
    'Parameter': [
        'Document unit',
        'min_df',
        'max_df',
        'Stopwords',
        'Lemmatization',
        'Vocab selection',
        'Vocab size cap',
        'LSTM layers (η)',
        'LSTM hidden (η)',
        'θ hidden dim',
        'Gradient clipping',
        'LR annealing',
    ],
    'Original (adjidieng/DETM)': [
        'Paragraphs (split speeches)',
        '10',
        '0.7',
        'stops.txt (~600 generic English)',
        'None',
        'All words passing df gates',
        'Unlimited',
        '3',
        '200',
        '800',
        '0.0 (off)',
        'Yes (÷4 after 10 bad epochs)',
    ],
    'Ours (updated)': [
        'Paragraphs (split speeches)',
        str(config.MIN_DF),
        str(config.MAX_DF),
        f'NLTK + {len(preprocessor.UN_STOPWORDS)} UN procedural words only',
        'None',
        'All words passing df gates',
        'Unlimited',
        str(config.LSTM_LAYERS),
        str(config.LSTM_HIDDEN),
        str(config.DOC_HIDDEN_DIM),
        str(config.CLIP_GRAD),
        'No',
    ],
    'Status': [
        '✓ Fixed',
        '✓ Fixed (was 30)',
        '✓ Fixed (was 0.3)',
        '✓ Fixed (was 40+ words)',
        '✓ Fixed (was WordNetLemmatizer)',
        '✓ Fixed (was top-10000 by IDF)',
        '✓ Fixed (was capped at 10000)',
        '○ Minor diff',
        '○ Minor diff (larger capacity)',
        '○ Same',
        '○ Minor diff',
        '◐ Moderate — affects convergence',
    ]
})

print("=" * 70)
print("PREPROCESSING COMPARISON: OURS (UPDATED) vs. ORIGINAL DETM")
print("=" * 70)
display(comparison.set_index('Parameter'))


### Recommendations to improve topic quality

**Must-fix (high impact):**

1. **Add paragraph splitting** — The original DETM splits each UN speech into paragraphs
   before building BoW. Each paragraph is treated as a separate document. This is the single
   largest difference: a paragraph about climate change will have a focused vocabulary,
   while a full speech mixes climate + security + economics in one bag-of-words vector.

2. **Relax `MAX_DF` from 0.3 → 0.7** — Our aggressive 0.3 threshold removes words appearing
   in >30% of documents. In the UN corpus, topically important words like "development",
   "peace", "security" appear very frequently but are essential topic markers. The original
   uses 0.7.

3. **Reduce UN-specific stopwords** — Words like "international", "world", "global",
   "countries", "government" were added to our custom stopword list but are NOT removed
   in the original. These words carry topic discrimination signal (e.g. "global warming",
   "international trade"). Remove or drastically reduce `UN_STOPWORDS`.

**Should-fix (moderate impact):**

4. **Lower `MIN_DF` from 30 → 10** — Matches the original's data preprocessing script.
   Domain-specific words like "ozone", "deforestation" may appear in fewer documents.

5. **Remove lemmatization** — The original does NOT lemmatize. Lemmatization can
   over-collapse meaningful distinctions (e.g. "developing" vs "development" may carry
   different topic signals).

6. **Remove IDF ranking / vocab cap** — The original keeps ALL words passing df gates
   without an IDF-based cap. Let the df gates alone control vocabulary size.

**After preprocessing fixes, the model must be retrained from scratch.**

---
## 14. Next Steps and Extensions

### Possible extensions:
1. **Temporal Analysis**: Track how specific topics evolve year-by-year
2. **Country-level Analysis**: Compare topic distributions across countries
3. **Dynamic Coherence**: Measure coherence changes over time
4. **Transfer to Financial Data**: Adapt this implementation for financial text
5. **Hyperparameter Optimization**: Grid search or Bayesian optimization
6. **Comparison with Baselines**: LDA, Static ETM, etc.

### Paper Reproducibility Checklist:
- [ ] Match model architecture (encoder layers, dimensions)
- [ ] Use same hyperparameters (learning rate, batch size, epochs)
- [ ] Verify embedding quality (Word2Vec vs BERT comparison)
- [ ] Compare coherence scores with paper benchmarks
- [ ] Validate perplexity on similar datasets
- [ ] Qualitative topic inspection

### Notes:
- **Modular Design**: Each component (encoder, decoder, losses) is self-contained
- **Algorithm Boundaries**: Clear separation between generative model, inference network, and temporal dynamics
- **Reproducibility**: All random seeds set, configurations saved
- **Extensibility**: Easy to swap components (e.g., different transition models)